# 🔥 PyTorch desde cero — Guía didáctica y completa

**Diplomado de IA · CENIA**

Este notebook está pensado para alguien que **no sabe nada** de PyTorch. Vamos a avanzar de a poco: primero los conceptos (qué es, por qué existe, qué es una GPU), luego la herramienta fundamental (el *tensor*), después cómo PyTorch calcula derivadas automáticamente (*autograd*) y, finalmente, cómo se entrena una red neuronal de principio a fin.

## 🗺️ Índice

| # | Sección | Qué aprenderás |
|---|---------|----------------|
| 1 | ¿Qué es PyTorch y por qué usarlo? | Contexto, ventajas, comparación con NumPy y TensorFlow |
| 2 | Hardware: CPU, GPU y compañía | Qué es cada uno, cómo saber si tienes GPU, cómo activarla, qué implica `cuda` o `mps` |
| 3 | Tensores: la estructura de datos base | Creación, forma, tipos de dato, dispositivo |
| 4 | Operaciones con tensores | Aritmética, indexado, `.item()`, reshape, `cat`/`stack`, broadcasting, matmul |
| 5 | PyTorch vs NumPy | Equivalencias, conversión, memoria compartida |
| 6 | CPU vs GPU en la práctica | Mover tensores, benchmark, código agnóstico al dispositivo |
| 7 | Autograd: derivadas automáticas | `requires_grad`, `backward()`, grafo computacional |
| 8 | Descenso de gradiente "a mano" | Entrenar un modelo usando solo tensores y autograd |
| 9 | `torch.nn`: bloques de redes neuronales | Módulos, capas, activaciones, pérdidas, optimizadores |
| 10 | Regresión lineal con `torch.nn` | Las 5 líneas del ciclo de entrenamiento |
| 11 | `Dataset` y `DataLoader` | Cómo alimentar datos por lotes (*batches*) |
| 12 | Clasificación con una red multicapa (MLP) | Entrenamiento + validación + frontera de decisión |
| 13 | Sobreajuste y regularización | Dropout, weight decay, `train()` vs `eval()` |
| 14 | Guardar y cargar modelos | `state_dict`, checkpoints |
| 15 | Redes convolucionales (CNN) con FashionMNIST | Un ejemplo real de visión por computador |
| 16 | Buenas prácticas y errores comunes | Reproducibilidad, depuración de shapes |
| 17 | Ejercicios integradores, *cheat sheet* y próximos pasos | Para practicar y seguir aprendiendo |

## ⚙️ Cómo usar este notebook
- Ejecuta las celdas **en orden** (`Shift + Enter`). Muchas dependen de las anteriores.
- **No te limites a leer**: cambia números, rompe cosas, mira los errores. Así se aprende de verdad.
- Si no tienes GPU, no pasa nada: todo funciona en CPU. En **Google Colab** puedes activar una GPU gratis (la sección 2 explica cómo).

## 📐 Convenciones del material
- Después de cada celda de código hay un bloque **🔍 Qué hace este código** que explica cada línea o ejemplo y qué resultado esperar.
- El código sigue el estilo **PEP 8** (nombres en `snake_case`, líneas de máximo 79 caracteres, dos líneas en blanco entre funciones) y las funciones y clases llevan **docstring**.
- Los bloques **🧠 Idea clave** resumen lo más importante.
- Cada **✏️ Ejercicio** viene con su **solución** en la celda siguiente. Intenta resolverlo antes de mirarla.
- Los números aleatorios cambian de una máquina a otra, así que cuando digo "deberías ver..." me refiero al **comportamiento**, no a cifras idénticas.

### Instalación (si aún no tienes PyTorch)
Lo más fiable es usar el selector oficial de https://pytorch.org/get-started/locally/ (eliges tu sistema operativo y si tienes GPU NVIDIA, y te da el comando exacto). Para esta guía necesitamos también `matplotlib` y `torchvision`.

In [1]:
# Descomenta la línea siguiente SOLO si te faltan paquetes
# (en Google Colab ya viene todo instalado).
# !pip install torch torchvision matplotlib numpy

---
# 1️⃣ ¿Qué es PyTorch y por qué usarlo?

## ¿Qué es?
**PyTorch** es una biblioteca de código abierto para Python que sirve para hacer **cálculo numérico** y **aprendizaje profundo (deep learning)**. Fue creada por el laboratorio de IA de Meta (Facebook) y hoy es mantenida por la *PyTorch Foundation*, parte de la Linux Foundation. Es, hoy por hoy, la herramienta dominante en investigación en IA, y muy usada también en la industria.

Se puede resumir en **tres grandes ideas**:

1. **Tensores**: una estructura de datos tipo "array multidimensional" (como los arrays de NumPy) que además puede vivir en una **GPU** para calcular mucho más rápido.
2. **Autograd**: un sistema que **calcula derivadas automáticamente**. Entrenar una red neuronal consiste en minimizar un error, y para eso se necesitan derivadas (gradientes). PyTorch las calcula por ti.
3. **`torch.nn` y herramientas relacionadas**: capas, funciones de pérdida, optimizadores y utilidades para cargar datos. Son los "ladrillos" para construir redes neuronales.

## ¿Por qué usar PyTorch (y no solo NumPy)?
NumPy es excelente, pero para deep learning le faltan dos cosas fundamentales:

| Necesidad del deep learning | NumPy | PyTorch |
|---|---|---|
| Calcular en **GPU** (miles de veces más paralelismo) | ❌ No | ✅ Sí |
| **Derivadas automáticas** (gradientes) | ❌ No | ✅ Sí (`autograd`) |
| Capas, optimizadores, pérdidas ya hechas | ❌ No | ✅ Sí (`torch.nn`, `torch.optim`) |
| Arrays multidimensionales y álgebra lineal | ✅ Sí | ✅ Sí |

## Otras ventajas
- **Grafo dinámico ("define-by-run")**: el grafo de cálculo se construye *mientras ejecutas* tu código normal de Python. Puedes usar `if`, `for`, `print()` y depurar con las herramientas habituales. Esto lo hace muy intuitivo.
- **"Pythónico"**: se siente como escribir Python normal, no como aprender un lenguaje nuevo.
- **Ecosistema enorme**: `torchvision` (imágenes), `torchaudio` (audio), Hugging Face Transformers (lenguaje), PyTorch Lightning, etc. La mayoría de los papers nuevos publican su código en PyTorch.
- **Del laboratorio a producción**: `torch.compile`, TorchScript, ONNX, etc.

## PyTorch vs TensorFlow (muy breve)
Ambos son frameworks de deep learning de primer nivel. Históricamente, TensorFlow destacaba en despliegue y PyTorch en investigación y facilidad de uso. Hoy ambos son capaces, pero PyTorch domina en la comunidad académica y en modelos de lenguaje. Si aprendes bien PyTorch, aprender el otro es mucho más fácil.

> 🧠 **Idea clave:** PyTorch = *NumPy con superpoderes* (GPU + derivadas automáticas) + *una caja de herramientas para redes neuronales*.

In [2]:
import torch

print("Versión de PyTorch:", torch.__version__)

Versión de PyTorch: 2.14.1


#### 🔍 Qué hace este código

- `import torch` carga la biblioteca en tu sesión. A partir de aquí, todo lo de PyTorch se accede con el prefijo `torch.` (por ejemplo `torch.tensor`, `torch.randn`).
- `torch.__version__` es un texto con la versión instalada, por ejemplo `2.x.x`. Si termina en algo como `+cpu` o `+cu1xx`, ese sufijo indica **para qué hardware fue compilada** tu instalación (CPU solamente, o CUDA con cierta versión). Esto será importante en la sección 2.
- Si esta celda da `ModuleNotFoundError`, PyTorch no está instalado en el entorno que usa tu notebook: revisa la sección de instalación al inicio.

---
# 2️⃣ Hardware: CPU, GPU y compañía

Antes de programar, entendamos **dónde** se ejecutan los cálculos.

## 2.1 CPU (Central Processing Unit)
Es el "cerebro" general de tu computador. Tiene **pocos núcleos** (típicamente 4 a 32), pero cada uno es **muy potente y versátil**: puede hacer cualquier tarea (abrir tu navegador, ejecutar Python, gestionar archivos...) y toma decisiones complejas muy rápido.

## 2.2 GPU (Graphics Processing Unit)
Nació para dibujar gráficos de videojuegos, donde hay que calcular millones de píxeles a la vez. Tiene **miles de núcleos simples** que hacen la **misma operación sobre muchos datos en paralelo**.

### 🍳 Analogía de la cocina
- **CPU** = 8 chefs de alta cocina. Cada uno puede preparar cualquier plato complejo, pero son pocos.
- **GPU** = 5.000 ayudantes que solo saben hacer una tarea simple (por ejemplo, picar cebolla), pero la hacen **todos a la vez**.

Si tienes que preparar un banquete de 1 millón de cebollas picadas, ganan los 5.000 ayudantes. Si tienes que preparar un solo plato sofisticado, ganan los chefs.

## 2.3 ¿Por qué la GPU es ideal para la IA?
Las redes neuronales están hechas de **multiplicaciones de matrices enormes**, y cada elemento del resultado se calcula de forma independiente de los demás. Es exactamente el tipo de problema "mucho de lo mismo en paralelo" donde brilla la GPU. Entrenar un modelo grande en GPU puede ser de **10x a 100x o más** más rápido que en CPU.

## 2.4 Otros términos que verás
| Término | Qué es |
|---|---|
| **CUDA** | Plataforma de NVIDIA que permite usar sus GPUs para cálculo general. En PyTorch el dispositivo se llama `"cuda"`. |
| **ROCm** | Equivalente de AMD. Las versiones de PyTorch para ROCm también usan el nombre `"cuda"` en el código. |
| **MPS** (*Metal Performance Shaders*) | Aceleración para GPUs de los Mac con chip Apple Silicon (M1, M2, M3...). Dispositivo `"mps"`. |
| **TPU** | *Tensor Processing Unit*: chip de Google diseñado específicamente para redes neuronales. |
| **RAM** | Memoria del computador (la usa la CPU). |
| **VRAM** | Memoria **de la GPU**. Es limitada (8, 16, 24, 80 GB...). Si tu modelo o tus datos no caben, obtienes el temido error *"CUDA out of memory"*. |
| **Driver** | Software que permite al sistema operativo "hablar" con la GPU. Sin el driver correcto, PyTorch no puede usarla. |

## 2.5 ⚠️ Un detalle importante
La CPU y (en las GPU NVIDIA) la GPU tienen **memorias separadas**. Para que la GPU calcule algo, hay que **copiar** los datos desde la RAM a la VRAM. Ese traslado cuesta tiempo, así que en PyTorch intentamos mover los datos una vez y hacer muchos cálculos allá. Un tensor siempre "vive" en un dispositivo concreto, y **no puedes operar un tensor de CPU con uno de GPU**.

> 🧠 **Idea clave:** GPU = muchísimo paralelismo para operaciones simples sobre matrices. Por eso entrena redes neuronales rápido. PyTorch te permite elegir en qué dispositivo vive cada tensor.

## 2.6 Primer vistazo: ¿qué dispositivos ve PyTorch?
La siguiente celda le pregunta a PyTorch qué aceleradores tiene disponibles y elige el mejor.

In [ ]:
import torch

print("¿CUDA (NVIDIA) disponible? ->", torch.cuda.is_available())
print("¿MPS (Apple) disponible?   ->", torch.backends.mps.is_available())

if torch.cuda.is_available():
    print("Nombre de la GPU:", torch.cuda.get_device_name(0))
    propiedades = torch.cuda.get_device_properties(0)
    print(f"VRAM total: {propiedades.total_memory / 1e9:.1f} GB")

# Patrón estándar para elegir el mejor dispositivo disponible
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("\nUsaremos el dispositivo:", device)

#### 🔍 Qué hace este código

Vamos línea por línea:

1. `import torch`: carga PyTorch (ya lo habíamos hecho, pero es buena práctica que cada celda importe lo que usa).
2. `torch.cuda.is_available()`: devuelve `True` si PyTorch encuentra una GPU **NVIDIA (CUDA)** que pueda usar, y `False` si no. Es una pregunta de "sí/no".
3. `torch.backends.mps.is_available()`: lo mismo, pero para la GPU de un **Mac con Apple Silicon** (`mps`).
4. El primer `if`: **solo si hay CUDA**, imprime el nombre de la tarjeta (`get_device_name(0)`; el `0` es el índice de la primera GPU) y su memoria total. `total_memory` viene en *bytes*; al dividir por `1e9` lo pasamos a gigabytes. `:.1f` significa "un decimal".
5. El segundo bloque `if / elif / else` **elige el dispositivo** por prioridad: primero CUDA, si no hay entonces MPS, y si tampoco hay, la CPU. El resultado se guarda en la variable `device`, que usaremos en todo el notebook.
6. `torch.device("cuda")` crea un **objeto que representa un dispositivo**. No hace cálculo alguno: es solo una "etiqueta" que luego le pasamos a los tensores y modelos para decirles dónde vivir. (`"cuda"` equivale a `"cuda:0"`, la primera GPU; si tuvieras varias, podrías usar `"cuda:1"`, etc.)
7. El `print` final muestra con cuál te quedaste.

**Qué verás según tu equipo:** en un PC sin GPU NVIDIA, ni Mac Apple Silicon: `False`, `False` y `Usaremos el dispositivo: cpu`. Con una NVIDIA bien configurada: `True`, el nombre de tu tarjeta y `cuda`. En un Mac M1/M2/M3: `False` para CUDA, `True` para MPS y `mps`.

## 2.7 ¿Qué implica usar `cuda` o `mps` en vez de `cpu`?

Elegir un dispositivo no es solo cambiar una palabra: **cambia cómo se comporta tu programa**.

| | `cpu` | `cuda` | `mps` |
|---|---|---|---|
| **Hardware** | Cualquier computador | GPU NVIDIA (o AMD con ROCm) | GPU de Mac con chip Apple Silicon |
| **Memoria** | RAM | **VRAM separada** de la RAM: hay que *copiar* datos | Memoria *unificada* (compartida con la CPU) |
| **Requisitos** | Ninguno | Driver de NVIDIA + PyTorch compilado con CUDA | macOS 12.3 o superior + PyTorch para Mac |
| **Velocidad** | La más lenta en operaciones grandes | La más rápida en general | Mucho más rápida que la CPU, pero normalmente por debajo de una GPU NVIDIA potente |
| **Limitaciones** | Ninguna | VRAM limitada (error *out of memory*) | Algunas operaciones aún no están implementadas; no soporta `float64` |

### Consecuencias prácticas (lo que cambia en tu código)
1. **Debes mover explícitamente modelo y datos.** PyTorch **no usa la GPU automáticamente**: si no haces `.to(device)`, todo se ejecuta en CPU aunque tengas la mejor GPU del mundo.
2. **Todos los tensores de una operación deben estar en el mismo dispositivo.** Sumar un tensor de CPU con uno de `cuda` da `RuntimeError`.
3. **Mover datos cuesta tiempo.** Con `cuda`, cada copia RAM → VRAM tiene un costo; por eso movemos los datos una vez y calculamos mucho allá. (En `mps` la memoria es compartida, pero PyTorch igual los trata como dispositivos distintos y exige `.to("mps")`.)
4. **La ejecución es asíncrona.** Cuando mandas una operación a la GPU, Python **no espera** a que termine y sigue con la siguiente línea. Consecuencias: para medir tiempos hay que "sincronizar" (lo veremos en la sección 6) y un error en la GPU a veces aparece varias líneas después de donde ocurrió.
5. **Memoria limitada.** Si el modelo + los datos + los gradientes no caben en la VRAM, obtienes *CUDA out of memory*. En CPU eso rara vez pasa porque tienes mucha más RAM.
6. **Para volver a NumPy o matplotlib hay que volver a CPU**: `tensor.cpu().numpy()`. NumPy solo entiende memoria de CPU.
7. **En `mps`:** no uses `float64` (no está soportado, usa `float32`). Si alguna operación no está implementada, PyTorch lanzará un error; puedes pedirle que use la CPU como alternativa para esa operación definiendo la variable de entorno `PYTORCH_ENABLE_MPS_FALLBACK=1` antes de importar torch.
8. **Los resultados pueden diferir muy levemente** entre CPU y GPU (por el orden de las operaciones con decimales). No es un error.

## 2.8 ¿Cómo saber si tengo GPU?
Hay que distinguir **dos preguntas distintas**:

- **(A) ¿Mi computador tiene una GPU?** (pregunta sobre el hardware)
- **(B) ¿PyTorch puede usarla?** (pregunta sobre el software instalado)

### (A) ¿Tiene mi equipo una GPU?
| Dónde estás | Cómo comprobarlo |
|---|---|
| **Windows** | `Ctrl + Shift + Esc` → *Administrador de tareas* → pestaña **Rendimiento** → busca **GPU 0 / GPU 1**. Verás el modelo (por ejemplo "NVIDIA GeForce RTX ...") y su memoria. Si solo aparece "Intel UHD/Iris" o "AMD Radeon Graphics" integrada, no tienes una NVIDIA dedicada. |
| **Linux / terminal con NVIDIA** | Ejecuta `nvidia-smi` en la terminal. Si responde con una tabla (modelo, memoria, procesos), tienes una GPU NVIDIA **y** el driver instalado. Si dice "command not found", falta el driver (o no tienes GPU NVIDIA). También sirve `lspci | grep -i nvidia`. |
| **Mac** | Menú  → *Acerca de este Mac*. Si el chip dice **Apple M1, M2, M3, M4...** tienes GPU utilizable con `mps`. Los Mac antiguos con procesador Intel no. |
| **Google Colab** | Menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución*. Si el acelerador es "GPU" (por ejemplo T4), la tienes. |

### (B) ¿PyTorch puede usarla?
Ejecuta la siguiente celda. Es un pequeño "diagnóstico" que sirve también para depurar cuando algo no funciona.

In [ ]:
print("Versión de PyTorch       :", torch.__version__)
print("Compilado con CUDA       :", torch.version.cuda)
print("CUDA disponible          :", torch.cuda.is_available())
print("Cantidad de GPUs CUDA    :", torch.cuda.device_count())
print("MPS incluido en PyTorch  :", torch.backends.mps.is_built())
print("MPS disponible           :", torch.backends.mps.is_available())

#### 🔍 Qué hace este código

- `torch.version.cuda`: indica con qué versión de CUDA fue **compilada tu instalación de PyTorch**. Si imprime `None`, lo que instalaste es la **versión solo-CPU** (no podrá usar una GPU NVIDIA aunque tengas una).
- `torch.cuda.is_available()`: `True` solo si PyTorch tiene soporte CUDA **y** encuentra una GPU con driver funcionando.
- `torch.cuda.device_count()`: cuántas GPUs ve PyTorch (0 si ninguna).
- `torch.backends.mps.is_built()`: si tu PyTorch incluye soporte para MPS (Mac).
- `torch.backends.mps.is_available()`: si además el equipo y el sistema operativo permiten usarlo.

**Cómo leer el resultado (equipos NVIDIA):**

| `torch.version.cuda` | `cuda.is_available()` | Diagnóstico |
|---|---|---|
| `None` | `False` | Instalaste PyTorch **sin soporte CUDA**. Hay que reinstalarlo (ver 2.9). |
| un número (p. ej. `12.x`) | `False` | PyTorch sí tiene CUDA, pero no ve la GPU: falta el **driver NVIDIA**, está desactualizado, o simplemente el equipo no tiene GPU NVIDIA. |
| un número | `True` | ✅ Todo en orden. |

**Equipos Mac:** si `is_built()` es `True` pero `is_available()` es `False`, probablemente tu macOS es anterior a 12.3 o tu Mac no tiene chip Apple Silicon.

## 2.9 ¿Cómo "activo" el uso de la GPU?
Aquí también hay **dos pasos distintos**: (1) dejar el entorno **listo** para que PyTorch vea la GPU, y (2) **pedirle a PyTorch** que la use en tu código.

### Paso 1 — Habilitar la GPU en tu entorno
| Tu situación | Qué hacer |
|---|---|
| **Google Colab** | *Entorno de ejecución → Cambiar tipo de entorno de ejecución → Acelerador por hardware: GPU → Guardar*. El runtime se reinicia: vuelve a ejecutar las celdas. No hay que instalar nada. |
| **Kaggle Notebooks** | Panel derecho → *Settings → Accelerator* → elige una GPU. |
| **PC/Linux con GPU NVIDIA** | (1) Instala o actualiza el **driver de NVIDIA**. (2) Instala la **versión de PyTorch con CUDA**: ve a https://pytorch.org/get-started/locally/, elige tu sistema operativo, `Pip` y la plataforma de cómputo **CUDA** (no "CPU"), y copia el comando que te muestra. (3) Reinicia el kernel del notebook y ejecuta la celda de diagnóstico. *No necesitas instalar por separado el "CUDA Toolkit"*: las versiones oficiales de PyTorch ya incluyen lo necesario. |
| **Mac con Apple Silicon** | Necesitas macOS 12.3 o superior y una instalación de Python nativa para Apple Silicon (arm64). Con el `pip install torch` normal ya viene el soporte para `mps`. |
| **Servidor del diplomado / universidad** | Consulta con tus docentes cómo acceder a las GPUs del servidor (a veces hay que reservarlas o pedirlas con un comando). |

> 💡 **Si ya tienes una GPU NVIDIA pero `torch.version.cuda` es `None`:** desinstala la versión actual (`pip uninstall torch torchvision`) e instala de nuevo con el comando del selector oficial eligiendo CUDA. Después **reinicia el kernel** del notebook.

### Paso 2 — Usar la GPU en tu código
**PyTorch nunca usa la GPU por su cuenta.** Todo lo que crees queda en CPU por defecto, y tú decides qué mover. La regla es:

> **Mueve el modelo y cada lote de datos al mismo dispositivo con `.to(device)`.**

La siguiente celda lo demuestra paso a paso.

In [ ]:
# 1) Por defecto, un tensor se crea en la CPU
t_cpu = torch.ones(3)
print("Por defecto el tensor está en:", t_cpu.device)

# 2) Crear un tensor directamente en el dispositivo elegido
t_dev = torch.ones(3, device=device)
print("Creado con device=device    :", t_dev.device)

# 3) Un modelo también nace en CPU y hay que moverlo
modelo_demo = torch.nn.Linear(3, 1)
print("Modelo antes de .to(device) :", next(modelo_demo.parameters()).device)

modelo_demo = modelo_demo.to(device)
print("Modelo después de .to(device):", next(modelo_demo.parameters()).device)

#### 🔍 Qué hace este código

- **Paso 1:** `torch.ones(3)` crea un tensor `[1., 1., 1.]` y, como no indicamos nada, queda en la **CPU**. El atributo `.device` nos dice dónde vive.
- **Paso 2:** `torch.ones(3, device=device)` crea el mismo tensor pero **directamente en el dispositivo elegido** (`device` es la variable definida en 2.6). Es más eficiente que crearlo en CPU y luego copiarlo.
- **Paso 3:** `torch.nn.Linear(3, 1)` es una capa de red neuronal (la veremos en la sección 9). Sus parámetros no son un tensor suelto, así que para ver dónde viven usamos `next(modelo_demo.parameters())`, que toma el **primer parámetro** del modelo, y miramos su `.device`.
- `modelo_demo.to(device)` mueve **todos** los parámetros del modelo al dispositivo.

**Qué verás:** si tu `device` es `cpu`, todas las líneas dirán `cpu` (no hay nada que mover). Si es una GPU NVIDIA, las líneas "antes" dirán `cpu` y las demás `cuda:0` (el `:0` indica "primera GPU"). En un Mac, `mps:0`.

---
# 3️⃣ Tensores: la estructura de datos base

Un **tensor** es un contenedor de números organizado en una o más dimensiones. Es la generalización de lo que ya conoces:

| Nombre | Dimensiones | Ejemplo | Ejemplo real |
|---|---|---|---|
| **Escalar** | 0 | `5` | Un valor de pérdida |
| **Vector** | 1 | `[1, 2, 3]` | Una lista de características de un cliente |
| **Matriz** | 2 | `[[1,2],[3,4]]` | Una tabla de datos, una imagen en blanco y negro |
| **Tensor 3D** | 3 | — | Una imagen a color: `(canales, alto, ancho)` |
| **Tensor 4D** | 4 | — | Un lote de imágenes: `(batch, canales, alto, ancho)` |

> En PyTorch **todo es un tensor**: tus datos, los pesos de la red, los gradientes, las predicciones.

## Los tres atributos que siempre debes mirar
Cada tensor tiene:
- **`shape`**: su forma (cuántos elementos hay en cada dimensión).
- **`dtype`**: el tipo de dato de sus elementos (`float32`, `int64`, ...).
- **`device`**: dónde vive (`cpu`, `cuda`, `mps`).

> Cuando algo falla en PyTorch, el 90% de las veces el problema es uno de estos tres. Acostúmbrate a imprimirlos.

## 3.1 Tensores de 0, 1, 2 y 3 dimensiones

In [ ]:
# Escalar (0 dimensiones)
escalar = torch.tensor(7)
# Vector (1 dimensión)
vector = torch.tensor([1.0, 2.0, 3.0])
# Matriz (2 dimensiones)
matriz = torch.tensor([[1, 2, 3],
                       [4, 5, 6]])
# Tensor 3D
tensor3d = torch.tensor([[[1, 2], [3, 4]],
                         [[5, 6], [7, 8]]])

ejemplos = [("escalar", escalar), ("vector", vector),
            ("matriz", matriz), ("tensor3d", tensor3d)]

for nombre, t in ejemplos:
    print(f"{nombre:9s} | ndim={t.ndim} | shape={tuple(t.shape)} | "
          f"dtype={t.dtype} | device={t.device}")

#### 🔍 Qué hace este código

- `torch.tensor(...)` **convierte datos de Python (números o listas) en un tensor**. El nivel de corchetes `[]` define el número de dimensiones:
  - `7` (sin corchetes) → escalar, 0 dimensiones.
  - `[1.0, 2.0, 3.0]` (un nivel) → vector de 3 elementos.
  - `[[1, 2, 3], [4, 5, 6]]` (dos niveles) → matriz de 2 filas y 3 columnas.
  - Tres niveles → tensor 3D. Aquí `[[[1,2],[3,4]], [[5,6],[7,8]]]` son **2 bloques**, cada uno una matriz de 2×2.
- La lista `ejemplos` guarda pares (nombre, tensor) para poder recorrerlos con un `for`.
- En el `print`, `t.ndim` es el **número de dimensiones**, `tuple(t.shape)` la **forma**, `t.dtype` el **tipo** y `t.device` el **dispositivo**. La parte `{nombre:9s}` solo alinea el texto en 9 caracteres.

**Resultado esperado:**

| Tensor | ndim | shape | dtype |
|---|---|---|---|
| escalar | 0 | `()` (vacío: no tiene dimensiones) | `int64` (porque escribimos un entero) |
| vector | 1 | `(3,)` | `float32` (porque escribimos decimales) |
| matriz | 2 | `(2, 3)` | `int64` |
| tensor3d | 3 | `(2, 2, 2)` | `int64` |

Observa cómo PyTorch **deduce el dtype**: enteros → `int64`, decimales → `float32`. Todos están en `cpu` porque no dijimos otra cosa.

## 3.2 Formas de crear tensores
Normalmente no escribirás los números a mano. Estas son las funciones que más usarás. Las dividimos en tres grupos.

### Grupo A: tensores con valores constantes o secuencias

In [ ]:
print("zeros(2, 3):\n", torch.zeros(2, 3))
print("\nones(2, 3):\n", torch.ones(2, 3))
print("\nfull((2, 2), 7.0):\n", torch.full((2, 2), 7.0))
print("\narange(0, 10, 2):", torch.arange(0, 10, 2))
print("linspace(0, 1, 5):", torch.linspace(0, 1, 5))
print("\neye(3):\n", torch.eye(3))

#### 🔍 Qué hace este código

| Línea | Qué hace | Resultado |
|---|---|---|
| `torch.zeros(2, 3)` | Tensor de **2 filas y 3 columnas** lleno de **ceros**. Los argumentos son las dimensiones. | `[[0,0,0],[0,0,0]]` |
| `torch.ones(2, 3)` | Igual, pero lleno de **unos**. | `[[1,1,1],[1,1,1]]` |
| `torch.full((2, 2), 7.0)` | Tensor de forma `(2, 2)` lleno con el valor que elijas (aquí `7.0`). Observa que aquí la forma va entre paréntesis y el valor después. | `[[7,7],[7,7]]` |
| `torch.arange(0, 10, 2)` | Secuencia desde **0 hasta 10 (sin incluir el 10), de 2 en 2**. Funciona como `range` de Python, pero devuelve un tensor. | `[0, 2, 4, 6, 8]` |
| `torch.linspace(0, 1, 5)` | **5 números equiespaciados entre 0 y 1, incluyendo ambos extremos.** (A diferencia de `arange`, tú fijas *cuántos* puntos, no el paso.) | `[0.00, 0.25, 0.50, 0.75, 1.00]` |
| `torch.eye(3)` | **Matriz identidad** de 3×3: unos en la diagonal, ceros en el resto. | `[[1,0,0],[0,1,0],[0,0,1]]` |

Los `\n` dentro de los `print` solo insertan saltos de línea para que las matrices se vean ordenadas.

### Grupo B: tensores aleatorios
Se usan sobre todo para **inicializar los pesos** de una red y para generar datos de prueba.

In [ ]:
torch.manual_seed(0)  # fija la semilla: los "azares" serán repetibles

print("rand(2, 3):\n", torch.rand(2, 3))
print("\nrandn(2, 3):\n", torch.randn(2, 3))
print("\nrandint(0, 10, (2, 3)):\n", torch.randint(0, 10, (2, 3)))

#### 🔍 Qué hace este código

- `torch.manual_seed(0)`: los computadores generan números "pseudoaleatorios" a partir de una **semilla**. Al fijarla (aquí en `0`), cada vez que ejecutes la celda obtendrás **exactamente los mismos números**. Es clave para que tus experimentos sean reproducibles.
- `torch.rand(2, 3)`: matriz de 2×3 con números **uniformes entre 0 y 1** (el 1 no se incluye). Todos los valores en ese rango son igual de probables.
- `torch.randn(2, 3)`: matriz de 2×3 con números de una **distribución normal** (campana de Gauss) con **media 0 y desviación estándar 1**. Verás valores positivos y negativos, la mayoría entre -2 y 2. (La `n` es de *normal*.)
- `torch.randint(0, 10, (2, 3))`: matriz de 2×3 con **enteros** entre 0 y 9 (el 10 no se incluye). Aquí la forma sí va entre paréntesis.

No te fijes en los valores concretos, sino en las **formas** y los **rangos**.

### Grupo C: crear tensores "a imagen" de otro

In [ ]:
base = torch.randn(4, 5)

ceros = torch.zeros_like(base)
unos = torch.ones_like(base)
aleatorio = torch.rand_like(base)

print("base      :", base.shape, base.dtype)
print("zeros_like:", ceros.shape, ceros.dtype)
print("ones_like :", unos.shape, unos.dtype)
print("rand_like :", aleatorio.shape, aleatorio.dtype)

#### 🔍 Qué hace este código

- `base = torch.randn(4, 5)` crea una matriz de 4×5 con números normales.
- Las funciones que terminan en **`_like`** (`zeros_like`, `ones_like`, `rand_like`) crean un tensor **con la misma forma, el mismo dtype y el mismo dispositivo que otro tensor**, pero con contenido distinto: ceros, unos o aleatorios.
- Esto es muy útil cuando no quieres repetir la forma a mano (y evita errores de dispositivo: el nuevo tensor queda en el mismo lugar que `base`).
- Los cuatro `print` deben mostrar `torch.Size([4, 5])` y `torch.float32`: comprobamos que las copias "a imagen" conservan forma y tipo.

## 3.3 Tipos de dato (`dtype`)
| dtype | Uso típico |
|---|---|
| `torch.float32` (por defecto para decimales) | **El estándar en deep learning** |
| `torch.float64` | Alta precisión (rara vez necesaria; ocupa el doble) |
| `torch.float16` / `torch.bfloat16` | Entrenamiento rápido y con menos memoria ("precisión mixta") |
| `torch.int64` (por defecto para enteros) | **Etiquetas de clase**, índices |
| `torch.int32`, `torch.uint8` | Enteros más pequeños, imágenes (0–255) |
| `torch.bool` | Máscaras (verdadero/falso) |

¿Por qué `float32` y no `float64`? Porque ocupa la mitad de memoria, es más rápido en GPU, y para entrenar redes la precisión extra no suele hacer falta.

In [ ]:
a = torch.tensor([1, 2, 3])          # enteros -> int64
b = torch.tensor([1.0, 2.0, 3.0])    # decimales -> float32
print("dtype de a:", a.dtype, "| dtype de b:", b.dtype)

# Convertir entre tipos
c = a.float()             # a float32
d = b.to(torch.float64)   # a float64
e = b.long()              # a int64
print("a.float():", c.dtype, "| b.to(float64):", d.dtype,
      "| b.long():", e.dtype)

# Convertir a entero TRUNCA los decimales (no redondea)
print("tensor([1.7, 2.9, -1.7]).long() ->",
      torch.tensor([1.7, 2.9, -1.7]).long())

# Mezclar tipos: el resultado toma el tipo "más amplio"
print("int64 + float32 ->", (a + b).dtype)

#### 🔍 Qué hace este código

- `torch.tensor([1, 2, 3])` contiene solo enteros → PyTorch elige `int64`. `torch.tensor([1.0, 2.0, 3.0])` contiene decimales → `float32`. (Basta un solo decimal en la lista para que todo sea `float32`.)
- **Conversiones:** hay dos formas equivalentes. Los atajos `.float()`, `.long()`, `.int()`, `.double()`, `.bool()`, o el método general `.to(dtype)`. **No modifican el tensor original: devuelven uno nuevo.**
- **`.long()` trunca:** convierte a entero eliminando la parte decimal sin redondear. Por eso `1.7 → 1`, `2.9 → 2` y `-1.7 → -1` (se acerca a cero).
- **Promoción de tipos:** al sumar un `int64` con un `float32`, el resultado es `float32` (el tipo capaz de representar ambos). Sin embargo, algunas funciones (por ejemplo `CrossEntropyLoss`) **exigen** un tipo concreto y fallan si no coincide, por eso conviene controlar los dtype.

> ✏️ **Ejercicio 3.1** — Crea un tensor `x` de forma `(3, 4)` lleno de números aleatorios normales, e imprime su `shape`, `dtype` y `device`. Luego conviértelo a `float64` e imprime de nuevo el `dtype`.
>
> *Intenta resolverlo tú primero; la solución está en la celda de código que sigue.*

In [ ]:
# Solución
x = torch.randn(3, 4)
print("shape :", x.shape)
print("dtype :", x.dtype)
print("device:", x.device)

x64 = x.to(torch.float64)
print("dtype tras convertir:", x64.dtype)
print("dtype del original (sin cambios):", x.dtype)

#### 🔍 Qué hace este código

- `torch.randn(3, 4)` crea la matriz pedida con valores normales.
- Imprimimos los tres atributos fundamentales: debería salir `torch.Size([3, 4])`, `torch.float32` y `cpu`.
- `x.to(torch.float64)` **devuelve una copia** en doble precisión que guardamos en `x64`. Por eso al final comprobamos que el original `x` sigue siendo `float32`.

---
# 4️⃣ Operaciones con tensores

## 4.1 Aritmética elemento a elemento
Las operaciones `+ - * / **` actúan **elemento por elemento** (y en paralelo, aunque no lo veas).

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
y = torch.tensor([10.0, 20.0, 30.0])

print("x + y   =", x + y)
print("x * y   =", x * y)
print("x ** 2  =", x ** 2)
print("x + 5   =", x + 5)
print("exp(x)  =", torch.exp(x))
print("sqrt(x) =", torch.sqrt(x))

# Operaciones "in-place": terminan en guion bajo y modifican el original
x.add_(100)
print("tras x.add_(100):", x)

#### 🔍 Qué hace este código

Partimos con `x = [1, 2, 3]` e `y = [10, 20, 30]`.

| Línea | Qué hace | Resultado |
|---|---|---|
| `x + y` | Suma **posición por posición**: 1+10, 2+20, 3+30 | `[11, 22, 33]` |
| `x * y` | Multiplica posición por posición. ⚠️ **No es** el producto de matrices (eso es `@`, sección 4.6) | `[10, 40, 90]` |
| `x ** 2` | Eleva **cada** elemento al cuadrado | `[1, 4, 9]` |
| `x + 5` | Suma 5 a **cada** elemento (un número suelto se aplica a todos) | `[6, 7, 8]` |
| `torch.exp(x)` | Calcula $e^x$ para cada elemento | `[2.7183, 7.3891, 20.0855]` |
| `torch.sqrt(x)` | Raíz cuadrada de cada elemento | `[1.0000, 1.4142, 1.7321]` |

**Operaciones in-place:** los métodos que terminan en `_` (como `add_`) **modifican el tensor original** en lugar de crear uno nuevo. Aquí `x.add_(100)` cambia `x` a `[101, 102, 103]`. Ahorran memoria, pero úsalas con cuidado con autograd (sección 7), porque pueden destruir información que se necesita para calcular gradientes.

## 4.2 Indexado y slicing
Funciona igual que en listas de Python y NumPy. Los índices empiezan en 0.

In [ ]:
m = torch.arange(12).reshape(3, 4)
print(m)

print("\nm[0]       :", m[0])
print("m[:, 1]    :", m[:, 1])
print("m[1:, :2]  :\n", m[1:, :2])
print("m[-1, -1]  :", m[-1, -1])

# Máscaras booleanas: filtrar por condición
mascara = m > 5
print("\nMáscara (m > 5):\n", mascara)
print("Elementos > 5:", m[mascara])

#### 🔍 Qué hace este código

- `torch.arange(12).reshape(3, 4)` crea los números 0 al 11 y los acomoda en 3 filas × 4 columnas:

```
[[ 0,  1,  2,  3],
 [ 4,  5,  6,  7],
 [ 8,  9, 10, 11]]
```

- **Sintaxis `m[filas, columnas]`**. Dentro de cada posición, `:` significa "todos", `a:b` es el rango desde `a` hasta `b` sin incluir `b`, y los índices negativos cuentan desde el final.
  - `m[0]` → la **primera fila**: `[0, 1, 2, 3]`.
  - `m[:, 1]` → **todas las filas, columna 1**: `[1, 5, 9]` (la segunda columna).
  - `m[1:, :2]` → filas desde la 1 hasta el final y columnas 0 y 1: `[[4, 5], [8, 9]]`.
  - `m[-1, -1]` → **último elemento**: `11`.
- **Máscara booleana:** `m > 5` compara **cada** elemento con 5 y devuelve un tensor de `True`/`False` de la misma forma. Usar esa máscara como índice (`m[mascara]`) **selecciona solo los elementos donde es `True`**: `[6, 7, 8, 9, 10, 11]` (el resultado es siempre un vector 1D).

### ¿Qué es `.item()` y qué implica convertir un tensor en "número de Python"?
Un tensor de un solo elemento, como `m[2, 3]`, **sigue siendo un tensor** (un objeto de PyTorch), no un número normal de Python. El método **`.item()`** extrae su valor y lo entrega como un **número nativo de Python** (`int`, `float` o `bool`).

| | Tensor de un elemento | Número de Python (`.item()`) |
|---|---|---|
| Tipo | `torch.Tensor` | `int`, `float` o `bool` |
| Guarda `dtype`, `device`, historial de gradientes | ✅ Sí | ❌ No: es solo un número |
| Puede vivir en la GPU | ✅ Sí | ❌ No: siempre está en la memoria de la CPU |
| Se puede usar en cualquier biblioteca (`math`, listas, `print` con formato, JSON...) | ⚠️ A veces falla o es lento | ✅ Sí |

**Lo que implica convertir con `.item()`:**
1. **Pierdes la conexión con autograd.** El número resultante ya no "recuerda" cómo se calculó, así que no puedes derivar a través de él. Por eso se usa `.item()` para **registrar** valores (por ejemplo, la pérdida), no para hacer cálculos que luego quieras derivar.
2. **Si el tensor está en GPU**, PyTorch debe **copiar ese valor a la CPU y esperar** a que la GPU termine lo que estaba haciendo (sincronización). Una vez por iteración está bien; en bucles muy cortos y frecuentes puede frenar el programa.
3. **Solo funciona con tensores de un único elemento.** Con más elementos da error; en ese caso se usa `.tolist()`, que devuelve una lista de Python.
4. **Evita fugas de memoria al acumular métricas.** Si sumas `total += loss` (tensor), cada suma conserva el grafo de autograd de esa iteración en memoria; con `total += loss.item()` solo guardas un número.

In [ ]:
t = torch.tensor(3.5)
print("Tipo del tensor        :", type(t))

numero = t.item()
print("Tipo tras .item()      :", type(numero), "| valor:", numero)

# El dtype del tensor decide el tipo de Python que se obtiene
print("int64   ->", type(torch.tensor(4).item()))
print("float32 ->", type(torch.tensor(4.0).item()))
print("bool    ->", type(torch.tensor(True).item()))

# Con más de un elemento, .item() falla y se usa .tolist()
varios = torch.tensor([1, 2, 3])
try:
    varios.item()
except (RuntimeError, ValueError) as error:
    print("\nError esperado con .item():", error)
print(".tolist() ->", varios.tolist())

# El número obtenido ya no está conectado a autograd
w = torch.tensor(2.0, requires_grad=True)
perdida = (w * 3) ** 2
print("\nperdida (tensor)  :", perdida, "| requires_grad:",
      perdida.requires_grad)
valor = perdida.item()
print("valor (float)     :", valor, "| tipo:", type(valor))

#### 🔍 Qué hace este código

1. `torch.tensor(3.5)` es un tensor escalar; `type(t)` confirma que es un `torch.Tensor`. Tras `.item()` obtenemos un `float` normal de Python (`3.5`).
2. El **tipo de Python depende del dtype**: un tensor entero da `int`, uno decimal da `float` y uno booleano da `bool`.
3. `varios.item()` falla porque el tensor tiene 3 elementos y `.item()` solo sabe convertir **uno**. Usamos `try / except` para capturar el error e imprimirlo sin que se detenga el notebook. Para obtener todos los valores se usa `.tolist()`, que da `[1, 2, 3]`.
4. Último ejemplo: `perdida = (w * 3) ** 2` vale `(2·3)² = 36`. Como `w` tiene `requires_grad=True`, el tensor `perdida` **recuerda cómo se calculó** (`requires_grad: True`). Al hacer `perdida.item()` obtenemos el `float` `36.0`, que es **solo un número**: no tiene historial ni gradiente.

**Regla práctica:** usa `.item()` cuando necesites un número "de verdad" para imprimir, acumular métricas, guardar en una lista o graficar. Si necesitas seguir derivando, mantén el tensor.

## 4.3 Cambiar la forma: `reshape`, `unsqueeze`, `squeeze`, `permute`
Cambiar la forma **no cambia los datos**, solo cómo los interpretamos. El número total de elementos debe mantenerse.

In [ ]:
t = torch.arange(12)
print("Original:", t.shape)

print("reshape(3, 4)  :", t.reshape(3, 4).shape)
print("reshape(2, -1) :", t.reshape(2, -1).shape)

# unsqueeze añade una dimensión de tamaño 1; squeeze las elimina
v = torch.tensor([1, 2, 3])
print("\nv                  :", v.shape)
print("v.unsqueeze(0)     :", v.unsqueeze(0).shape)
print("v.unsqueeze(1)     :", v.unsqueeze(1).shape)
print("unsqueeze(0).squeeze():", v.unsqueeze(0).squeeze().shape)

# permute reordena dimensiones: imagen (alto, ancho, canales) -> (C, H, W)
img = torch.randn(32, 32, 3)
print("\nImagen HWC:", img.shape, "-> CHW:", img.permute(2, 0, 1).shape)

# Transpuesta de una matriz
M = torch.arange(6).reshape(2, 3)
print("M:", M.shape, "-> M.T:", M.T.shape)

# reshape NO copia los datos (cuando puede): comparten memoria
r = t.reshape(3, 4)
r[0, 0] = 100
print("\nTras modificar r[0, 0], t[0] vale:", t[0].item())

#### 🔍 Qué hace este código

- **`reshape(3, 4)`**: reacomoda los 12 elementos en 3 filas × 4 columnas. Siempre debe cumplirse que `3 × 4 = 12`.
- **`reshape(2, -1)`**: el `-1` significa *"calcula tú esta dimensión"*. Como hay 12 elementos y 2 filas, PyTorch deduce 6 columnas → `(2, 6)`.
- **`unsqueeze(d)`** agrega una dimensión de tamaño 1 en la posición `d`. Sirve para convertir un vector `(3,)` en una **fila** `(1, 3)` (con `unsqueeze(0)`) o en una **columna** `(3, 1)` (con `unsqueeze(1)`). Se usa constantemente para añadir la dimensión de *batch* o para preparar broadcasting.
- **`squeeze()`** hace lo contrario: **elimina todas las dimensiones de tamaño 1**. Por eso `unsqueeze(0).squeeze()` vuelve a `(3,)`.
- **`permute(2, 0, 1)`** reordena las dimensiones: la dimensión que estaba en la posición 2 pasa a ser la primera, etc. Aquí convierte una imagen en formato *alto × ancho × canales* `(32, 32, 3)` al formato que usa PyTorch, *canales × alto × ancho* `(3, 32, 32)`.
- **`M.T`** es la **transpuesta** (filas ↔ columnas): `(2, 3)` pasa a `(3, 2)`.
- **Memoria compartida:** `r = t.reshape(3, 4)` no copia los datos, solo ofrece otra "vista" de la misma memoria. Al cambiar `r[0, 0]` a 100, también cambia `t[0]`, y el último `print` muestra `100`. Si necesitas una copia independiente usa `.clone()`.

## 4.4 Reducciones: sumas, promedios, máximos...
Colapsan una dimensión. El argumento **`dim`** indica *cuál* dimensión desaparece.

In [ ]:
m = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])

print("sum total          :", m.sum())
print("sum(dim=0)         :", m.sum(dim=0))
print("sum(dim=1)         :", m.sum(dim=1))
print("mean / std         :", m.mean(), "/", m.std())
print("max                :", m.max())
print("argmax (aplanado)  :", m.argmax())
print("argmax(dim=1)      :", m.argmax(dim=1))
print("sum(dim=1) shape   :", m.sum(dim=1).shape)
print("con keepdim=True   :", m.sum(dim=1, keepdim=True).shape)

#### 🔍 Qué hace este código

Trabajamos con esta matriz de 2 filas × 3 columnas:

```
[[1, 2, 3],
 [4, 5, 6]]
```

- `m.sum()` suma **todos** los elementos → `21`.
- **`dim` indica la dimensión que "desaparece"**:
  - `sum(dim=0)` colapsa las **filas** (suma hacia abajo, columna por columna) → `[5, 7, 9]`.
  - `sum(dim=1)` colapsa las **columnas** (suma hacia el lado, fila por fila) → `[6, 15]`.
- `m.mean()` es el promedio (`3.5`) y `m.std()` la desviación estándar (`≈ 1.87`).
- `m.max()` entrega el valor máximo (`6`). **`argmax()`** entrega en cambio **la posición** del máximo: sin `dim`, cuenta sobre el tensor "aplanado" (`[1,2,3,4,5,6]`), así que da `5`. Con `dim=1` entrega, **para cada fila, la columna donde está su máximo**: `[2, 2]`. (`argmax(dim=1)` es lo que usaremos para obtener la clase predicha por una red.)
- `keepdim=True` conserva la dimensión colapsada con tamaño 1: el resultado pasa de `(2,)` a `(2, 1)`. Es útil para luego combinarlo por broadcasting con el tensor original.

## 4.5 Concatenar y apilar: `cat` y `stack`
- `torch.cat`: une tensores **a lo largo de una dimensión que ya existe**.
- `torch.stack`: une tensores **creando una dimensión nueva**.

In [ ]:
a = torch.zeros(2, 3)
b = torch.ones(2, 3)

print("cat   dim=0:", torch.cat([a, b], dim=0).shape)
print("cat   dim=1:", torch.cat([a, b], dim=1).shape)
print("stack dim=0:", torch.stack([a, b], dim=0).shape)

#### 🔍 Qué hace este código

Tenemos `a` (una matriz 2×3 de ceros) y `b` (una matriz 2×3 de unos). Ambas se pasan **dentro de una lista**.

- `cat(..., dim=0)`: las **pega una debajo de la otra** (a lo largo de las filas) → `(4, 3)`: 4 filas y 3 columnas.
- `cat(..., dim=1)`: las **pega una al lado de la otra** (a lo largo de las columnas) → `(2, 6)`.
- `stack(..., dim=0)`: crea una **dimensión nueva** al inicio y coloca cada tensor como un "piso" → `(2, 2, 3)`. Es como apilar dos hojas una sobre otra: ahora tienes un tensor con 2 matrices de 2×3.

**Diferencia clave:** `cat` conserva el número de dimensiones (sigue siendo 2D); `stack` lo aumenta en uno (pasa a 3D).

### ¿Deben tener las mismas dimensiones los tensores que quiero unir?
**Depende de la función**, y es una de las fuentes más frecuentes de errores:

| Función | Condición para poder unir |
|---|---|
| **`torch.cat`** | • Deben tener el **mismo número de dimensiones** (no puedes unir una matriz con un vector). <br>• El tamaño debe ser **igual en todas las dimensiones excepto en la dimensión por la que unes**, que sí puede ser distinta. <br>• Deben estar en el **mismo dispositivo** (y conviene que tengan el mismo `dtype`). |
| **`torch.stack`** | Deben tener **exactamente la misma forma**, sin excepciones (porque se apilan como pisos idénticos). Además, mismo dispositivo. |

Por ejemplo, una matriz `(2, 3)` y otra `(2, 4)` **sí** se pueden pegar *lado a lado* (`dim=1`) porque coinciden en las filas (2), y la dimensión de unión (columnas) puede diferir → resultado `(2, 7)`. Pero **no** se pueden pegar *una debajo de otra* (`dim=0`), porque al unir por filas las columnas deben coincidir (3 ≠ 4).

In [ ]:
a = torch.zeros(2, 3)
c = torch.full((2, 4), 2.0)
d = torch.ones(5, 3)
vector = torch.ones(3)

# Casos que SÍ funcionan
print("cat([a, c], dim=1):", torch.cat([a, c], dim=1).shape)
print("cat([a, d], dim=0):", torch.cat([a, d], dim=0).shape)

# Casos que FALLAN: capturamos el error para ver el mensaje
casos_con_error = {
    "cat([a, c], dim=0)": lambda: torch.cat([a, c], dim=0),
    "cat([a, vector])": lambda: torch.cat([a, vector], dim=0),
    "stack([a, c])": lambda: torch.stack([a, c]),
}
for nombre, funcion in casos_con_error.items():
    try:
        funcion()
    except RuntimeError as error:
        print(f"\n{nombre} -> {type(error).__name__}: {error}")

# Uso práctico: añadir una columna de unos (el "truco del sesgo")
con_sesgo = torch.cat([a, torch.ones(2, 1)], dim=1)
print("\nMatriz (2,3) + columna de unos ->", con_sesgo.shape)

#### 🔍 Qué hace este código

- **Casos que funcionan:**
  - `cat([a, c], dim=1)`: `a` es `(2, 3)` y `c` es `(2, 4)`. Coinciden en las filas (2) y se unen por columnas → `(2, 7)`.
  - `cat([a, d], dim=0)`: `a` es `(2, 3)` y `d` es `(5, 3)`. Coinciden en las columnas (3) y se unen por filas → `(7, 3)`.
- **Casos que fallan** (guardados en un diccionario de funciones `lambda` para poder probarlos uno por uno dentro de `try / except`):
  - `cat([a, c], dim=0)`: al unir por filas, las columnas deben ser iguales (3 vs 4) → `RuntimeError` indicando que los tamaños deben coincidir excepto en la dimensión de unión.
  - `cat([a, vector])`: `a` tiene 2 dimensiones y `vector` solo 1 → no se pueden unir (distinto número de dimensiones).
  - `stack([a, c])`: `stack` exige formas idénticas y `(2, 3) ≠ (2, 4)` → error.
- **Uso práctico:** `torch.cat([a, torch.ones(2, 1)], dim=1)` agrega una columna de unos a la derecha de `a`, pasando de `(2, 3)` a `(2, 4)`. Es una técnica clásica para incluir el término independiente (sesgo) dentro de una multiplicación de matrices.

> 🧠 Si `cat` o `stack` fallan, imprime el `.shape` de cada tensor: casi siempre verás de inmediato cuál dimensión no calza.

## 4.6 Multiplicación de matrices (`matmul`, `@`)
Es **la** operación del deep learning. Para multiplicar $A$ de forma $(n, k)$ con $B$ de forma $(k, m)$, la dimensión interna ($k$) debe coincidir, y el resultado es $(n, m)$.

Cada capa lineal de una red neuronal hace básicamente: $\text{salida} = \text{entrada} \cdot W + b$.

In [ ]:
A = torch.randn(2, 3)
B = torch.randn(3, 4)
C = A @ B
print("A:", A.shape, "| B:", B.shape, "| A @ B:", C.shape)

# Producto punto entre dos vectores
u = torch.tensor([1.0, 2.0, 3.0])
w = torch.tensor([4.0, 5.0, 6.0])
print("\nu · w =", torch.dot(u, w).item())

# Diferencia entre * (elemento a elemento) y @ (producto matricial)
P = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("\nP * P:\n", P * P)
print("P @ P:\n", P @ P)

# Si las dimensiones no calzan, PyTorch avisa
try:
    torch.randn(2, 3) @ torch.randn(2, 3)
except RuntimeError as error:
    print("\nError esperado:", error)

#### 🔍 Qué hace este código

- `A @ B` es el operador de multiplicación matricial (equivale a `torch.matmul(A, B)`). Con `A` de `(2, 3)` y `B` de `(3, 4)`, la dimensión interna es 3 en ambas ✅, y el resultado tiene la forma exterior: `(2, 4)`.
- `torch.dot(u, w)` es el **producto punto** de dos vectores: `1·4 + 2·5 + 3·6 = 32`. Usamos `.item()` para mostrarlo como número.
- **`*` vs `@`** con la matriz `P = [[1, 2], [3, 4]]`:
  - `P * P` multiplica **elemento a elemento**: `[[1, 4], [9, 16]]`.
  - `P @ P` es el **producto matricial** (fila por columna): `[[7, 10], [15, 22]]`. Por ejemplo, el elemento (0,0) es `1·1 + 2·3 = 7`.
- En el último ejemplo multiplicamos `(2, 3) @ (2, 3)`: la dimensión interna sería 3 y 2, que no coinciden → `RuntimeError` ("shapes cannot be multiplied"). Es el error de forma más común en redes neuronales.

## 4.7 Broadcasting (transmisión)
Permite operar tensores de **formas distintas** sin copiar datos manualmente. PyTorch "estira" virtualmente la dimensión más pequeña.

**Regla:** se comparan las formas **de derecha a izquierda**. Dos dimensiones son compatibles si son **iguales** o si **una de ellas es 1** (o no existe).

```
Matriz (3, 4)  +  Vector (4,)   ->  el vector se repite en cada fila  -> (3, 4)
Columna (3, 1) +  Fila (1, 4)   ->  ambos se estiran                  -> (3, 4)
```

In [ ]:
M = torch.arange(12.0).reshape(3, 4)
fila = torch.tensor([100.0, 200.0, 300.0, 400.0])
col = torch.tensor([[1.0], [2.0], [3.0]])

print("M + fila:\n", M + fila)
print("\ncol + fila.unsqueeze(0):\n", col + fila.unsqueeze(0))

# Un caso incompatible: (3, 4) + (3,) -> se comparan 4 y 3
try:
    M + torch.ones(3)
except RuntimeError as error:
    print("\nError esperado:", error)

# Aplicación: normalizar datos columna a columna
escalas = torch.tensor([1.0, 10.0, 100.0])
desplazamientos = torch.tensor([0.0, 50.0, -20.0])
datos = torch.randn(100, 3) * escalas + desplazamientos

normalizados = (datos - datos.mean(dim=0)) / datos.std(dim=0)
print("\nMedias tras normalizar :", normalizados.mean(dim=0))
print("Desv. est. tras normalizar:", normalizados.std(dim=0))

#### 🔍 Qué hace este código

- **`M + fila`**: `M` es `(3, 4)` y `fila` es `(4,)`. Comparando de derecha a izquierda: 4 = 4 ✅, y a `fila` le "falta" una dimensión, así que se repite en cada fila de `M`. Resultado: a cada fila de `M` se le suma `[100, 200, 300, 400]` → primera fila `[100, 201, 302, 403]`.
- **`col + fila.unsqueeze(0)`**: `col` es `(3, 1)` y `fila.unsqueeze(0)` es `(1, 4)`. En cada dimensión una de las dos vale 1, así que **ambas se estiran** hasta `(3, 4)`. Resultado: una "tabla de sumas" donde la entrada (i, j) es `col[i] + fila[j]` (primera fila `[101, 201, 301, 401]`).
- **Caso incompatible:** `(3, 4) + (3,)` compara `4` con `3`: no son iguales ni ninguno es 1 → error. Para que funcionara, el vector tendría que ser una columna `(3, 1)`: `M + torch.ones(3).unsqueeze(1)`.
- **Aplicación real, normalizar datos:**
  1. Creamos 100 muestras con 3 características de escalas muy distintas (`escalas` y `desplazamientos` también se aplican por broadcasting).
  2. `datos.mean(dim=0)` calcula **el promedio de cada columna** → forma `(3,)`.
  3. `datos - promedio` resta a cada fila ese vector (broadcasting); luego se divide por la desviación de cada columna.
  4. El resultado tiene **media ≈ 0 y desviación ≈ 1 en cada columna** (verás medias del orden de 1e-8, es decir, cero con error numérico). Esta es la normalización "estándar" previa a casi cualquier entrenamiento.

> ✏️ **Ejercicio 4.1** — Crea una matriz `A` de forma `(5, 3)` con valores aleatorios. (a) Calcula el promedio de cada columna. (b) Réstalo a `A` usando broadcasting. (c) Verifica que el promedio de cada columna ahora es ≈ 0.
>
> *Intenta resolverlo tú primero; la solución está en la celda de código que sigue.*

In [ ]:
# Solución
A = torch.randn(5, 3)

# (a) Promedio de cada columna: colapsamos las filas con dim=0
medias = A.mean(dim=0)
print("Medias (forma", tuple(medias.shape), "):", medias)

# (b) Restamos con broadcasting: (5, 3) - (3,) -> (5, 3)
A_centrada = A - medias
print("Forma de A_centrada:", tuple(A_centrada.shape))

# (c) Las nuevas medias deben ser ≈ 0
nuevas_medias = A_centrada.mean(dim=0)
print("Nuevas medias:", nuevas_medias)
print("¿Todas ≈ 0?:", torch.allclose(nuevas_medias, torch.zeros(3),
                                     atol=1e-5))

#### 🔍 Qué hace este código

- (a) `A.mean(dim=0)` colapsa las 5 filas y deja **un promedio por columna**: forma `(3,)`.
- (b) `A - medias` resta a cada fila de `A` ese vector. Funciona por broadcasting: comparando de derecha a izquierda, `3 = 3` ✅ y a `medias` le falta la dimensión de las filas, que se "estira".
- (c) Volvemos a promediar por columnas: salen números minúsculos (≈ 1e-8). `torch.allclose(a, b, atol=1e-5)` devuelve `True` si **todos** los elementos de `a` y `b` difieren en menos de la tolerancia `atol`. Comparar decimales con `==` es mala idea por los errores de redondeo; `allclose` es la forma correcta.

---
# 5️⃣ PyTorch vs NumPy

Si ya usaste NumPy, PyTorch te resultará muy familiar. De hecho, un tensor se parece muchísimo a un `ndarray`.

## Equivalencias más comunes
| NumPy | PyTorch | Nota |
|---|---|---|
| `np.array([1,2,3])` | `torch.tensor([1,2,3])` | |
| `np.zeros((2,3))` | `torch.zeros(2, 3)` | PyTorch acepta la forma sin paréntesis extra |
| `np.random.randn(2,3)` | `torch.randn(2, 3)` | |
| `a.shape` | `t.shape` | Igual |
| `a.dtype` | `t.dtype` | Los dtypes tienen otro nombre: `torch.float32` |
| `a.reshape(...)` | `t.reshape(...)` | Igual |
| `a.T` | `t.T` | Igual |
| `np.concatenate` | `torch.cat` | |
| `np.stack` | `torch.stack` | |
| `a @ b`, `np.dot` | `t @ u`, `torch.matmul` | |
| `a.sum(axis=0)` | `t.sum(dim=0)` | **`axis` → `dim`** (PyTorch también acepta `axis`) |
| `np.expand_dims(a, 0)` | `t.unsqueeze(0)` | |
| `a.astype(np.float32)` | `t.to(torch.float32)` / `t.float()` | |

## Diferencias fundamentales
| | NumPy | PyTorch |
|---|---|---|
| Dispositivo | Solo CPU | CPU **y GPU** |
| Derivadas automáticas | No | **Sí** (`requires_grad`) |
| Tipo decimal por defecto | `float64` | `float32` |
| Propósito | Cálculo científico general | Deep learning (y cálculo general) |
| Ecosistema de redes neuronales | No tiene | `torch.nn`, `torch.optim`, ... |

In [ ]:
import numpy as np

# La misma operación en ambos
a_np = np.array([[1, 2], [3, 4]], dtype=np.float32)
a_pt = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)

print("NumPy   :", (a_np @ a_np).tolist())
print("PyTorch :", (a_pt @ a_pt).tolist())

# Diferencia de dtype por defecto
print("\nDtype por defecto en NumPy  :", np.array([1.0]).dtype)
print("Dtype por defecto en PyTorch:", torch.tensor([1.0]).dtype)

#### 🔍 Qué hace este código

- Creamos la **misma matriz** `[[1, 2], [3, 4]]` como `ndarray` de NumPy (`a_np`) y como tensor de PyTorch (`a_pt`), ambas en `float32` (el argumento `dtype=` lo fuerza).
- `a_np @ a_np` y `a_pt @ a_pt` ejecutan el **mismo producto matricial** con la misma sintaxis. `.tolist()` convierte el resultado en una lista de Python para imprimirlo de forma idéntica. Ambos dan `[[7.0, 10.0], [15.0, 22.0]]`.
- Al final comparamos los **dtype por defecto**: sin indicar nada, NumPy usa `float64` y PyTorch `float32`. Esta diferencia importa cuando conviertes datos entre ambas bibliotecas (muchos errores vienen de ahí).

## Convertir entre NumPy y PyTorch
- `torch.from_numpy(arr)` → de NumPy a tensor.
- `tensor.numpy()` → de tensor a NumPy (solo si el tensor está en **CPU** y **no requiere gradiente**; si no, primero `.detach().cpu()`).

### ⚠️ Memoria compartida
`from_numpy` y `.numpy()` **no copian los datos**: ambos objetos apuntan a la **misma memoria**. Si modificas uno, cambia el otro. Esto es eficiente, pero puede sorprenderte.

In [ ]:
arr = np.ones(3)
t_compartido = torch.from_numpy(arr)
t_copia = torch.tensor(arr)

arr[0] = 99
print("arr          :", arr)
print("t_compartido :", t_compartido)
print("t_copia      :", t_copia)

# De tensor a NumPy: también comparten memoria
t = torch.tensor([1.0, 2.0, 3.0])
n = t.numpy()
t[0] = -1
print("\ntensor:", t, "| numpy:", n)

# Copiar vs compartir dentro de PyTorch
original = torch.tensor([1, 2, 3])
alias = original              # el mismo objeto con otro nombre
copia = original.clone()      # copia independiente
original[0] = 100
print("\nalias:", alias, "| copia:", copia)

# Si el tensor está en GPU o tiene gradiente, la forma segura es:
# arr = t.detach().cpu().numpy()

#### 🔍 Qué hace este código

1. `np.ones(3)` crea el array `[1., 1., 1.]`.
2. `torch.from_numpy(arr)` crea un tensor **que comparte memoria** con `arr`. `torch.tensor(arr)` en cambio **copia** los datos.
3. Al hacer `arr[0] = 99`, el array cambia, y **también `t_compartido`** (verás `99.` en su primera posición), mientras que `t_copia` conserva `[1., 1., 1.]`. (Los tres son `float64` porque NumPy usa `float64` por defecto.)
4. `t.numpy()` hace lo mismo en la dirección contraria: `n` es una vista de la memoria de `t`. Al cambiar `t[0]` a `-1`, `n` también refleja `-1.`.
5. **Copiar vs compartir en PyTorch:** `alias = original` **no copia nada**, es el mismo objeto con otro nombre; por eso, al modificar `original`, `alias` cambia. `original.clone()` sí crea una **copia independiente** (queda en `[1, 2, 3]`).
6. El comentario final recuerda la receta segura para pasar a NumPy cualquier tensor: `.detach()` (corta el vínculo con autograd), `.cpu()` (lo trae de la GPU) y recién entonces `.numpy()`.

## Comparación de velocidad en CPU
En CPU, NumPy y PyTorch rinden de forma muy similar en operaciones simples (ambos usan bibliotecas de álgebra lineal optimizadas). La **gran** diferencia aparece cuando entra la GPU (siguiente sección).

In [ ]:
import time


def medir(funcion, repeticiones=5):
    """Mide el tiempo promedio (en segundos) de ejecutar ``funcion``.

    Primero hace una ejecución de calentamiento, que no se cuenta, y
    luego promedia ``repeticiones`` ejecuciones.

    Args:
        funcion: función sin argumentos que se quiere medir.
        repeticiones: cuántas veces se mide.

    Returns:
        Tiempo promedio por ejecución, en segundos.
    """
    funcion()
    inicio = time.perf_counter()
    for _ in range(repeticiones):
        funcion()
    return (time.perf_counter() - inicio) / repeticiones


n = 2000
A_np = np.random.rand(n, n).astype(np.float32)
B_np = np.random.rand(n, n).astype(np.float32)
A_pt = torch.from_numpy(A_np)
B_pt = torch.from_numpy(B_np)

t_numpy = medir(lambda: A_np @ B_np)
t_torch = medir(lambda: A_pt @ B_pt)
print(f"NumPy   (CPU), matmul {n}x{n}: {t_numpy:.4f} s")
print(f"PyTorch (CPU), matmul {n}x{n}: {t_torch:.4f} s")

#### 🔍 Qué hace este código

- **`medir(funcion, repeticiones)`** es una pequeña utilidad que definimos nosotros. Recibe **una función sin argumentos**, la ejecuta una vez para "calentar" (la primera ejecución suele ser más lenta por inicializaciones) y luego mide cuánto demoran `repeticiones` ejecuciones usando `time.perf_counter()`, un reloj de alta precisión. Devuelve el promedio.
- Creamos dos matrices de 2000×2000 con números aleatorios en NumPy (`.astype(np.float32)` las pasa a `float32` para que la comparación sea justa) y las convertimos a tensores con `torch.from_numpy` (sin copiar, comparten memoria).
- `lambda: A_np @ B_np` crea una función anónima sin argumentos que realiza la multiplicación; así podemos pasársela a `medir`.
- Verás tiempos **del mismo orden de magnitud** para ambas bibliotecas (los valores exactos dependen de tu CPU).

---
# 6️⃣ CPU vs GPU en la práctica

## 6.1 Mover tensores entre dispositivos
- `tensor.to(device)` → devuelve una copia en el dispositivo indicado.
- `tensor.cuda()` / `tensor.cpu()` → atajos.
- También puedes **crear** el tensor directamente en el dispositivo: `torch.randn(3, 3, device=device)`.

> Recuerda: **todos los tensores de una operación deben estar en el mismo dispositivo.**

In [ ]:
x_cpu = torch.randn(3, 3)
x_dev = x_cpu.to(device)
print("x_cpu está en:", x_cpu.device, "| x_dev está en:", x_dev.device)

# Crear directamente en el dispositivo (más eficiente)
y_dev = torch.randn(3, 3, device=device)

# Operar tensores en el mismo dispositivo: funciona
print("x_dev + y_dev está en:", (x_dev + y_dev).device)

# .to() sobre un tensor NO lo modifica: devuelve uno nuevo
t = torch.ones(2)
t.to(device)
print("\nTras t.to(device) sin asignar, t sigue en:", t.device)
t = t.to(device)
print("Tras t = t.to(device), t está en:", t.device)

# Mezclar dispositivos: error (solo ocurre si tienes GPU)
if device.type != "cpu":
    try:
        x_cpu + y_dev
    except RuntimeError as error:
        print("\nError esperado al mezclar dispositivos:\n", error)

#### 🔍 Qué hace este código

- `x_cpu.to(device)` crea una **copia** de `x_cpu` en el dispositivo elegido en la sección 2 (`device`). Si `device` es `cpu`, no hay nada que copiar y devuelve el mismo tensor.
- `torch.randn(3, 3, device=device)` crea el tensor ya en el dispositivo, sin pasar por la CPU.
- `x_dev + y_dev` funciona porque **ambos** están en el mismo dispositivo.
- **Trampa clásica con tensores:** `t.to(device)` **no cambia `t`**, devuelve un tensor nuevo. Si no lo guardas (`t = t.to(device)`), `t` sigue donde estaba. (Con los *modelos* de `torch.nn` es distinto: `modelo.to(device)` sí modifica el modelo en el lugar, aunque se suele escribir igual con asignación.)
- El último bloque **solo se ejecuta si tienes GPU** (`device.type != "cpu"`): intenta sumar un tensor de CPU con uno de GPU y muestra el error típico *"Expected all tensors to be on the same device"*.

## 6.2 Benchmark: ¿cuánto más rápida es la GPU?
**Importante:** la GPU ejecuta las operaciones de forma **asíncrona** (tu código Python sigue corriendo mientras la GPU trabaja). Para medir tiempos correctamente hay que **sincronizar** (esperar a que termine) antes de detener el reloj. Y conviene hacer una ejecución de "calentamiento" porque la primera vez incluye costos de inicialización.

In [ ]:
def sincronizar(dispositivo):
    """Espera a que la GPU termine las operaciones pendientes.

    En CPU no hace nada. Es necesario para medir tiempos correctamente,
    porque la GPU trabaja de forma asíncrona.

    Args:
        dispositivo: objeto ``torch.device`` donde se ejecutan los cálculos.
    """
    if dispositivo.type == "cuda":
        torch.cuda.synchronize()
    elif dispositivo.type == "mps":
        torch.mps.synchronize()


def benchmark_matmul(dispositivo, n=2048, repeticiones=3):
    """Mide el tiempo promedio de multiplicar dos matrices n x n.

    Args:
        dispositivo: ``torch.device`` donde se hará la multiplicación.
        n: tamaño de las matrices cuadradas.
        repeticiones: número de multiplicaciones que se promedian.

    Returns:
        Tiempo promedio por multiplicación, en segundos.
    """
    a = torch.randn(n, n, device=dispositivo)
    b = torch.randn(n, n, device=dispositivo)
    _ = a @ b                      # calentamiento
    sincronizar(dispositivo)
    inicio = time.perf_counter()
    for _ in range(repeticiones):
        _ = a @ b
    sincronizar(dispositivo)
    return (time.perf_counter() - inicio) / repeticiones


t_cpu = benchmark_matmul(torch.device("cpu"))
print(f"CPU: {t_cpu:.4f} s por multiplicación (2048x2048)")

if device.type != "cpu":
    t_gpu = benchmark_matmul(device)
    print(f"{device.type.upper()}: {t_gpu:.4f} s por multiplicación")
    print(f"👉 La GPU fue ~{t_cpu / t_gpu:.1f} veces más rápida")
else:
    print("No hay GPU disponible. Prueba este notebook en Google Colab "
          "con GPU para ver la diferencia.")

#### 🔍 Qué hace este código

- **`sincronizar(dispositivo)`**: llama a `torch.cuda.synchronize()` (o su equivalente de MPS), que **bloquea Python hasta que la GPU haya terminado todo lo que tenía pendiente**. En CPU no hace nada. Sin esto, el reloj se detendría apenas Python *envía* la orden, no cuando la GPU la *termina*, y mediríamos tiempos irreales (muy cortos).
- **`benchmark_matmul(dispositivo, n, repeticiones)`**:
  1. Crea dos matrices aleatorias de `n × n` **directamente en el dispositivo**.
  2. `_ = a @ b` es la ejecución de calentamiento; se guarda en `_` porque no nos interesa el resultado.
  3. Sincroniza, arranca el reloj, repite la multiplicación, vuelve a sincronizar y calcula el promedio.
- Primero medimos en CPU. Si hay GPU (`device.type != "cpu"`), medimos también allí e imprimimos **cuántas veces más rápida** resultó. Con una GPU decente suele ser un factor de 10x o mucho más.
- Si estás solo en CPU verás el mensaje que sugiere probar en Colab con GPU.

In [ ]:
# Si tienes una GPU NVIDIA, así puedes ver cuánta memoria está usando
if device.type == "cuda":
    asignada = torch.cuda.memory_allocated() / 1e6
    reservada = torch.cuda.memory_reserved() / 1e6
    print(f"Memoria asignada a tensores: {asignada:.1f} MB")
    print(f"Memoria reservada por PyTorch: {reservada:.1f} MB")
else:
    print("Esta celda solo aplica a GPUs NVIDIA (cuda).")

#### 🔍 Qué hace este código

- `torch.cuda.memory_allocated()` indica cuántos *bytes* de VRAM ocupan actualmente tus tensores; lo dividimos por `1e6` para verlo en megabytes.
- `torch.cuda.memory_reserved()` es la memoria que PyTorch **le pidió a la GPU** (suele ser mayor, porque reserva con anticipación para ser más eficiente).
- Estas cifras son útiles para entender por qué aparece el error *out of memory*. Fuera de Python, el comando `nvidia-smi` en la terminal muestra el uso de la GPU en tiempo real.

## 6.3 🧰 El patrón "código agnóstico al dispositivo"
Escribe tu código **una sola vez** de modo que funcione igual en CPU o GPU:

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo = MiModelo().to(device)         # mueve los parámetros del modelo
for x, y in loader:
    x, y = x.to(device), y.to(device)  # mueve cada lote de datos
    ...
```

Para pasar un resultado de GPU a NumPy o matplotlib: `t.detach().cpu().numpy()`.

> 🧠 **Idea clave:** la GPU brilla con operaciones **grandes**. Para tensores minúsculos, el costo de copiar datos puede hacerla *más lenta* que la CPU.

---
# 7️⃣ Autograd: derivadas automáticas

Esta es la **función estrella** de PyTorch. Para entender por qué importa, repasemos cómo "aprende" una red neuronal:

1. Hacemos una **predicción** con los parámetros actuales.
2. Medimos qué tan mal lo hicimos con una **función de pérdida** (*loss*).
3. Calculamos el **gradiente**: la derivada de la pérdida respecto a cada parámetro. Nos dice *en qué dirección y cuánto* cambiar cada parámetro para reducir el error.
4. **Actualizamos** los parámetros dando un pequeño paso en la dirección contraria al gradiente (*descenso de gradiente*).

Una red moderna puede tener millones o miles de millones de parámetros. Calcular esas derivadas a mano es imposible. **Autograd lo hace por nosotros**, usando la **regla de la cadena** sobre el historial de operaciones.

## ¿Cómo funciona?
Cuando un tensor tiene `requires_grad=True`, PyTorch **registra cada operación** que le haces, formando un **grafo computacional** (un "árbol genealógico" de cálculos). Al llamar `.backward()` sobre el resultado final, recorre ese grafo **hacia atrás** aplicando la regla de la cadena y deja el gradiente de cada parámetro en su atributo `.grad`.

```
x  ──►  [ x² ]  ──►  [ +2x ]  ──►  [ +1 ]  ──►  y
                                                 │
                         .backward()  ◄──────────┘   (recorre el grafo al revés)
x.grad = dy/dx
```

In [ ]:
# y = x² + 2x + 1  ->  dy/dx = 2x + 2.  En x = 3, la derivada vale 8.
x = torch.tensor(3.0, requires_grad=True)
print("Antes de backward, x.grad =", x.grad)

y = x ** 2 + 2 * x + 1
print("y =", y.item())
print("grad_fn de y:", y.grad_fn)

y.backward()
print("Después de backward, x.grad =", x.grad.item())

#### 🔍 Qué hace este código

- `torch.tensor(3.0, requires_grad=True)` crea un tensor con valor 3 y le pide a PyTorch que **registre todo lo que se haga con él**, para poder derivar después.
- `x.grad` está vacío (`None`) al principio: todavía no calculamos ninguna derivada.
- `y = x ** 2 + 2 * x + 1` es la función $y = x^2 + 2x + 1$. Evaluada en $x = 3$ da $9 + 6 + 1 = 16$. Mientras se calcula, PyTorch va construyendo el grafo de operaciones.
- `y.grad_fn` muestra **la última operación registrada** (algo como `<AddBackward0 ...>`: la suma final). Es la prueba de que `y` "recuerda" de dónde viene.
- `y.backward()` recorre el grafo hacia atrás aplicando la regla de la cadena y deposita en `x.grad` la derivada $\frac{dy}{dx}$ evaluada en $x = 3$. Analíticamente es $2x + 2 = 8$, y eso es exactamente lo que imprime.

### Con varias variables
El gradiente es un vector con la derivada respecto a **cada** entrada.
Ejemplo: $f(a, b) = a^2 \cdot b + b^3$. Entonces $\frac{\partial f}{\partial a} = 2ab$ y $\frac{\partial f}{\partial b} = a^2 + 3b^2$.

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)

f = a ** 2 * b + b ** 3
f.backward()

print("df/da =", a.grad.item(), "(esperado: 2*a*b =", 2 * 2 * 3, ")")
print("df/db =", b.grad.item(), "(esperado: a² + 3b² =",
      2 ** 2 + 3 * 3 ** 2, ")")

#### 🔍 Qué hace este código

- Declaramos dos variables, `a = 2` y `b = 3`, ambas con seguimiento de gradiente.
- `f = a ** 2 * b + b ** 3` vale `4·3 + 27 = 39`.
- Con **una sola llamada** `f.backward()`, PyTorch calcula **las derivadas respecto a ambas variables** y las deja en `a.grad` y `b.grad`.
- Comparamos con las fórmulas que derivamos a mano: $\partial f/\partial a = 2ab = 12$ y $\partial f/\partial b = a^2 + 3b^2 = 4 + 27 = 31$. Coinciden. En una red real, esto mismo ocurre simultáneamente con millones de parámetros.

## ⚠️ Los gradientes se **acumulan**
Cada vez que llamas a `.backward()`, el gradiente nuevo se **suma** al que ya había en `.grad`. Por eso, en cada iteración de entrenamiento hay que **ponerlos a cero** (`optimizer.zero_grad()` o `tensor.grad.zero_()`). Es uno de los errores más comunes de principiantes.

In [ ]:
w = torch.tensor(1.0, requires_grad=True)

(w * 2).backward()
print("Tras el 1er backward:", w.grad.item())

(w * 2).backward()
print("Tras el 2do backward (acumulado):", w.grad.item())

w.grad.zero_()
print("Tras zero_():", w.grad.item())

#### 🔍 Qué hace este código

- La derivada de `w * 2` respecto a `w` es siempre `2`.
- Después del primer `backward()`, `w.grad` vale `2`, como esperábamos.
- Al llamar `backward()` **otra vez** sin limpiar, el nuevo `2` **se suma** al anterior: `w.grad` vale `4`, no `2`.
- `w.grad.zero_()` (in-place, por el guion bajo) lo reinicia a `0`. En un entrenamiento real, si olvidas este paso en cada iteración, los gradientes de iteraciones pasadas contaminan los actuales y el modelo aprende mal.

## Desactivar el seguimiento de gradientes
Seguir el grafo cuesta memoria y tiempo. Cuando **no** vas a entrenar (por ejemplo, al hacer predicciones o evaluar), conviene apagarlo:

- `with torch.no_grad():` → bloque donde no se registran operaciones.
- `tensor.detach()` → devuelve un tensor "desconectado" del grafo.
- `@torch.no_grad()` → decorador equivalente para funciones completas.
- `torch.inference_mode()` → versión aún más optimizada para inferencia.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y = x * 3
print("Operación normal       -> requires_grad:", y.requires_grad)

with torch.no_grad():
    z = x * 3
print("Dentro de no_grad      -> requires_grad:", z.requires_grad)

w = (x * 3).detach()
print("Después de detach()    -> requires_grad:", w.requires_grad)

#### 🔍 Qué hace este código

- `y = x * 3` se hace "con seguimiento": `y` hereda `requires_grad=True` y guardará su historial → imprime `True`.
- Dentro del bloque `with torch.no_grad():`, la misma operación **no se registra**: `z` tiene `requires_grad=False` → imprime `False`. Es lo que se usa al evaluar un modelo, para ahorrar memoria y tiempo.
- `.detach()` toma un tensor que sí estaba en el grafo y devuelve una versión **desconectada**, sin gradiente → `False`. Es lo que hacemos antes de pasar un resultado a NumPy o a matplotlib.

> 🧠 **Idea clave:** `requires_grad=True` + `.backward()` = derivadas gratis. Recuerda: los gradientes se acumulan (hay que ponerlos a cero), y se apagan con `no_grad()` cuando solo evalúas.

> ✏️ **Ejercicio 7.1** — Sea $f(x) = \sin(x) \cdot x^2$. Calcula con autograd $f'(\pi)$ y compáralo con la derivada analítica $f'(x) = \cos(x)\,x^2 + 2x\sin(x)$.
>
> *Intenta resolverlo tú primero; la solución está en la celda de código que sigue.*

In [ ]:
# Solución
import math

x = torch.tensor(math.pi, requires_grad=True)
f = torch.sin(x) * x ** 2
f.backward()

analitica = (math.cos(math.pi) * math.pi ** 2
             + 2 * math.pi * math.sin(math.pi))

print("Derivada con autograd :", x.grad.item())
print("Derivada analítica    :", analitica)
print("Diferencia absoluta   :", abs(x.grad.item() - analitica))

#### 🔍 Qué hace este código

- Creamos `x = π` como tensor con seguimiento de gradiente (`math.pi` es el valor de π en Python).
- `f = torch.sin(x) * x ** 2` evalúa la función. Al llamar `f.backward()`, autograd aplica **regla del producto y regla de la cadena** automáticamente y deja $f'(\pi)$ en `x.grad`.
- La derivada que calculamos a mano es $\cos(\pi)\pi^2 + 2\pi\sin(\pi)$. Como $\cos(\pi) = -1$ y $\sin(\pi) = 0$, vale $-\pi^2 \approx -9.8696$.
- Ambos resultados coinciden hasta unos 6 decimales. La pequeña diferencia se debe a que PyTorch calcula en `float32` y Python en `float64`. Moraleja: **no tuvimos que derivar nada a mano para obtener el gradiente**.

---
# 8️⃣ Descenso de gradiente "a mano"

Ahora juntamos todo: vamos a **entrenar un modelo usando solo tensores y autograd**, sin `torch.nn`. Así verás exactamente qué ocurre "bajo el capó".

**Problema:** tenemos datos generados con $y = 3x + 2$ más algo de ruido, y queremos que el modelo **descubra** los valores 3 y 2.

**Modelo:** $\hat{y} = w \cdot x + b$, con parámetros $w$ y $b$ que parten aleatorios.

**Pérdida (MSE, error cuadrático medio):** $L = \frac{1}{N}\sum (\hat{y}_i - y_i)^2$

**Actualización:** $w \leftarrow w - \eta \cdot \frac{\partial L}{\partial w}$, donde $\eta$ es la **tasa de aprendizaje** (*learning rate*): qué tan grande es cada paso.

In [ ]:
import matplotlib.pyplot as plt

torch.manual_seed(42)

# Datos sintéticos: y = 3x + 2 + ruido
x = torch.linspace(-3, 3, 100)
y = 3 * x + 2 + torch.randn(100) * 0.5

# Parámetros: parten en valores arbitrarios y necesitan gradiente
w = torch.randn(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

lr = 0.05
historial = []

for epoca in range(100):
    y_pred = w * x + b                  # 1) forward
    loss = ((y_pred - y) ** 2).mean()   # 2) pérdida (MSE)
    loss.backward()                     # 3) backward

    with torch.no_grad():               # 4) actualización
        w -= lr * w.grad
        b -= lr * b.grad

    w.grad.zero_()                      # 5) limpiar gradientes
    b.grad.zero_()

    historial.append(loss.item())
    if epoca % 20 == 0:
        print(f"Época {epoca:3d} | loss = {loss.item():.4f} | "
              f"w = {w.item():.3f} | b = {b.item():.3f}")

print(f"\nResultado: w = {w.item():.3f} (real: 3), "
      f"b = {b.item():.3f} (real: 2)")

#### 🔍 Qué hace este código

**Preparación**
- `torch.manual_seed(42)` fija la aleatoriedad para que sea reproducible.
- `x = torch.linspace(-3, 3, 100)`: 100 puntos equiespaciados entre -3 y 3.
- `y = 3 * x + 2 + torch.randn(100) * 0.5`: la "verdad" ($3x + 2$) más **ruido normal** de desviación 0.5, para simular datos reales imperfectos.
- `w` y `b` son los **parámetros que el modelo debe aprender**. `w` parte aleatorio, `b` parte en 0, y ambos llevan `requires_grad=True` porque queremos derivar respecto a ellos.
- `lr = 0.05` es la tasa de aprendizaje (el tamaño de cada paso) y `historial` guardará la pérdida de cada época para graficarla.

**Cada vuelta del `for` (una "época")** hace los 5 pasos que resumen *todo* el entrenamiento de redes neuronales:
1. **Forward:** `w * x + b` calcula la predicción para los 100 puntos a la vez (los operadores trabajan elemento a elemento).
2. **Pérdida:** `((y_pred - y) ** 2).mean()` es el error cuadrático medio: qué tan lejos están las predicciones de la realidad.
3. **Backward:** `loss.backward()` calcula $\partial L/\partial w$ y $\partial L/\partial b$ y los deja en `w.grad` y `b.grad`.
4. **Actualización:** `w -= lr * w.grad` mueve cada parámetro **un paso en contra del gradiente**. Se hace dentro de `torch.no_grad()` porque esta modificación **no debe registrarse** en el grafo (y PyTorch prohíbe modificar in-place un parámetro con seguimiento fuera de ese bloque).
5. **Limpieza:** `.grad.zero_()`, porque los gradientes se acumulan.

- `loss.item()` extrae la pérdida como número de Python para guardarla (ver sección 4.2).
- `if epoca % 20 == 0` imprime solo cada 20 épocas (`%` es el resto de la división).

**Qué esperar:** la pérdida debería bajar rápido y estabilizarse cerca de `0.25` (la varianza del ruido, $0.5^2$: no puede ser menor porque el ruido es impredecible), y al final `w ≈ 3` y `b ≈ 2` (no exactos por el ruido, pero cercanos).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(historial)
ax[0].set_title("La pérdida baja con el entrenamiento")
ax[0].set_xlabel("Época")
ax[0].set_ylabel("MSE")

ax[1].scatter(x.numpy(), y.numpy(), s=10, label="datos")
ax[1].plot(x.numpy(), (w * x + b).detach().numpy(), color="red",
           label="modelo aprendido")
ax[1].set_title("Ajuste final")
ax[1].legend()
plt.show()

#### 🔍 Qué hace este código

- `plt.subplots(1, 2, ...)` crea una figura con **1 fila y 2 gráficos** (`ax[0]` y `ax[1]`).
- **Gráfico izquierdo:** la curva de `historial`: debería ser decreciente y aplanarse (el modelo converge).
- **Gráfico derecho:** los puntos de datos (`scatter`) y, en rojo, la recta que aprendió el modelo. Debería atravesar la nube de puntos.
- Para graficar con matplotlib convertimos los tensores con `.numpy()`; la recta se calcula con `w * x + b`, que **tiene seguimiento de gradiente**, así que primero usamos `.detach()` (receta de la sección 5).

> 🧠 **Idea clave:** *todo* el entrenamiento de redes neuronales —incluso GPT— es esta misma receta: **forward → loss → backward → actualizar → zero_grad**, repetida muchas veces. Lo demás son detalles de escala.

> ✏️ **Ejercicio 8.1** — Repite el entrenamiento con `lr = 0.001` y luego con `lr = 0.5`. ¿Qué ocurre en cada caso? (Pista: observa la curva de pérdida. ¿Converge lento? ¿Diverge?)
>
> *Intenta resolverlo tú primero; la solución está en la celda de código que sigue.*

In [ ]:
# Solución
def entrenar_manual(x, y, lr, n_epocas=30, semilla=42):
    """Entrena y = w*x + b con descenso de gradiente escrito a mano.

    Args:
        x: tensor 1D con las entradas.
        y: tensor 1D con los valores objetivo.
        lr: tasa de aprendizaje.
        n_epocas: número de épocas de entrenamiento.
        semilla: semilla para inicializar ``w`` de forma reproducible.

    Returns:
        Lista con la pérdida (MSE) obtenida en cada época.
    """
    torch.manual_seed(semilla)
    w = torch.randn(1, requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    perdidas = []
    for _ in range(n_epocas):
        loss = ((w * x + b - y) ** 2).mean()
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
        perdidas.append(loss.item())
    return perdidas


plt.figure(figsize=(7, 4))
for tasa in [0.001, 0.05, 0.5]:
    perdidas = entrenar_manual(x, y, lr=tasa)
    print(f"lr = {tasa:<6} -> pérdida inicial {perdidas[0]:.2f}, "
          f"final {perdidas[-1]:.3e}")
    plt.plot(perdidas, label=f"lr = {tasa}")

plt.yscale("log")
plt.xlabel("Época")
plt.ylabel("MSE (escala logarítmica)")
plt.title("Efecto de la tasa de aprendizaje")
plt.legend()
plt.show()

#### 🔍 Qué hace este código

- Convertimos el ciclo de la celda anterior en una **función reutilizable** `entrenar_manual` que recibe la tasa de aprendizaje y devuelve el historial de pérdidas. Usamos solo 30 épocas para que, en el caso malo, los números no se desborden.
- El `for tasa in [0.001, 0.05, 0.5]` entrena tres veces con cada tasa y las dibuja juntas. La escala logarítmica (`plt.yscale("log")`) permite ver a la vez curvas de magnitudes muy distintas.

**Qué observarás:**
- **`lr = 0.001` (muy pequeña):** la pérdida baja, pero **muy lentamente**; en 30 épocas apenas se mueve. Es seguro, pero ineficiente.
- **`lr = 0.05` (adecuada):** baja rápido y se estabiliza cerca de 0.25 (el nivel de ruido).
- **`lr = 0.5` (demasiado grande):** cada paso **sobrepasa** el mínimo y rebota cada vez más lejos: la pérdida **crece sin control** (diverge, llega a valores gigantescos). Con estos datos el límite aproximado es `lr ≈ 0.33`; por encima de eso, el entrenamiento se vuelve inestable.

> 🧠 La tasa de aprendizaje es el **hiperparámetro más importante** al entrenar: ni muy pequeña (lento) ni muy grande (diverge).

---
# 9️⃣ `torch.nn`: bloques para construir redes neuronales

Hacer todo a mano no escala. `torch.nn` nos da componentes reutilizables.

## Piezas principales
| Pieza | Módulo | Ejemplos |
|---|---|---|
| **Capas** | `torch.nn` | `nn.Linear`, `nn.Conv2d`, `nn.LSTM`, `nn.Embedding` |
| **Funciones de activación** | `torch.nn` | `nn.ReLU`, `nn.Sigmoid`, `nn.Tanh`, `nn.GELU` |
| **Funciones de pérdida** | `torch.nn` | `nn.MSELoss` (regresión), `nn.CrossEntropyLoss` (clasificación), `nn.BCEWithLogitsLoss` (binaria) |
| **Optimizadores** | `torch.optim` | `optim.SGD`, `optim.Adam`, `optim.AdamW` |

## ¿Por qué necesitamos activaciones?
Si apilas capas lineales sin nada entre ellas, el conjunto sigue siendo **una sola transformación lineal** (por mucho que apiles). Las **activaciones no lineales** (como ReLU: `max(0, x)`) son lo que permite a la red aprender relaciones complejas.

## `nn.Module`: la clase base de todo
Todo modelo en PyTorch es una clase que **hereda de `nn.Module`** y define:
- `__init__`: crea las capas (aquí "declaras" qué piezas tiene).
- `forward(x)`: define cómo fluye la entrada por las capas (aquí "conectas" las piezas).

PyTorch registra automáticamente los parámetros de todas las capas, de modo que `model.parameters()` los entrega al optimizador, y autograd calcula sus gradientes.

In [ ]:
import torch.nn as nn

# Una capa lineal: salida = entrada · Wᵀ + b
capa = nn.Linear(in_features=3, out_features=2)
print("Pesos W:", tuple(capa.weight.shape))
print("Sesgo b:", tuple(capa.bias.shape))
print("¿Requieren gradiente?", capa.weight.requires_grad)

x = torch.randn(5, 3)
print("Forma de la salida:", tuple(capa(x).shape))

# Funciones de activación
z = torch.linspace(-3, 3, 7)
print("\nz          :", z)
print("ReLU(z)    :", nn.ReLU()(z))
print("Sigmoid(z) :", nn.Sigmoid()(z))
print("Tanh(z)    :", nn.Tanh()(z))

#### 🔍 Qué hace este código

**La capa `nn.Linear(3, 2)`**
- Recibe vectores de **3 características** y entrega vectores de **2 valores**. Internamente guarda una matriz de pesos `W` de forma `(2, 3)` y un vector de sesgos `b` de forma `(2,)`, inicializados al azar. Calcula `entrada @ Wᵀ + b`.
- Los parámetros ya vienen con `requires_grad=True` (imprime `True`): son lo que la red va a aprender.
- `x = torch.randn(5, 3)` simula un **lote de 5 ejemplos** con 3 características cada uno. `capa(x)` → forma `(5, 2)`: 5 ejemplos, ahora con 2 valores cada uno. **La capa se usa como si fuera una función.**
- Regla de oro: la **última dimensión de la entrada** debe ser igual a `in_features`.

**Activaciones** (aplicadas a `z = [-3, -2, -1, 0, 1, 2, 3]`):
| Función | Qué hace | Resultado aproximado |
|---|---|---|
| `ReLU` | Pone en 0 los negativos y deja pasar los positivos | `[0, 0, 0, 0, 1, 2, 3]` |
| `Sigmoid` | "Aplasta" cada valor al rango (0, 1); útil para probabilidades | `[0.05, 0.12, 0.27, 0.50, 0.73, 0.88, 0.95]` |
| `Tanh` | Aplasta al rango (-1, 1) | `[-0.995, -0.964, -0.762, 0, 0.762, 0.964, 0.995]` |

Todas actúan **elemento por elemento** y no tienen parámetros que aprender.

### Definir un modelo: dos maneras
**1) `nn.Sequential`** — para cadenas simples de capas.
**2) Subclase de `nn.Module`** — máxima flexibilidad (la forma estándar en proyectos reales).

In [ ]:
# Manera 1: Sequential (las capas se aplican en el orden dado)
modelo_seq = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Linear(16, 3),
)


# Manera 2: subclase de nn.Module
class MLP(nn.Module):
    """Perceptrón multicapa con dos capas ocultas.

    Args:
        n_entrada: número de características de entrada.
        n_oculta: número de neuronas de cada capa oculta.
        n_salida: número de valores de salida (por ejemplo, clases).
    """

    def __init__(self, n_entrada, n_oculta, n_salida):
        super().__init__()
        self.capa1 = nn.Linear(n_entrada, n_oculta)
        self.capa2 = nn.Linear(n_oculta, n_oculta)
        self.capa3 = nn.Linear(n_oculta, n_salida)
        self.act = nn.ReLU()

    def forward(self, x):
        """Calcula la salida (logits) para un lote ``x``."""
        x = self.act(self.capa1(x))
        x = self.act(self.capa2(x))
        return self.capa3(x)


modelo = MLP(2, 16, 3)
print(modelo)

n_params = sum(p.numel() for p in modelo.parameters())
print("\nTotal de parámetros entrenables:", n_params)

for nombre, parametro in modelo.named_parameters():
    print(f"{nombre:15s} {tuple(parametro.shape)}")

# Se usa como una función (¡no llames a .forward() directamente!)
lote = torch.randn(4, 2)
print("\nSalida para un lote de 4 ejemplos:", tuple(modelo(lote).shape))

#### 🔍 Qué hace este código

**Manera 1: `nn.Sequential`.** Se le pasan las capas en orden y el modelo las aplica una tras otra: `2 → 16 → ReLU → 16 → ReLU → 3`. Es ideal cuando los datos fluyen en línea recta.

**Manera 2: clase propia que hereda de `nn.Module`:**
- `__init__` **declara las piezas**: tres capas lineales y una activación. `super().__init__()` es obligatorio y debe ir **primero**: inicializa la maquinaria de `nn.Module` que registra los parámetros.
- `forward(self, x)` define **cómo pasan los datos por las piezas**: capa 1 → ReLU → capa 2 → ReLU → capa 3. Devolvemos los valores crudos (*logits*), sin softmax, porque la función de pérdida de clasificación lo hace por dentro.
- Los docstrings explican qué hace la clase y el método (buena práctica en PEP 8 / PEP 257).

**Usar el modelo:**
- `print(modelo)` muestra la arquitectura.
- `sum(p.numel() for p in modelo.parameters())` cuenta los parámetros: `numel()` es el número de elementos de cada tensor. Para este modelo: capa1 `2·16 + 16 = 48`, capa2 `16·16 + 16 = 272`, capa3 `16·3 + 3 = 51`, total **371**.
- `named_parameters()` lista cada parámetro con su nombre y forma: por ejemplo `capa1.weight (16, 2)` y `capa1.bias (16,)`.
- `modelo(lote)` con un lote de 4 ejemplos de 2 características devuelve forma `(4, 3)`: 4 ejemplos × 3 salidas. Siempre llama al modelo con paréntesis (`modelo(x)`), **no** con `.forward(x)`: así PyTorch ejecuta además sus mecanismos internos (hooks, etc.).

## Funciones de pérdida y optimizadores
- **`nn.MSELoss`** → regresión (predecir números).
- **`nn.CrossEntropyLoss`** → clasificación multiclase. **Recibe *logits* crudos** (sin softmax) y **etiquetas enteras** (tipo `int64`) con el índice de la clase. Internamente aplica softmax + log + NLL.
- **`nn.BCEWithLogitsLoss`** → clasificación binaria (también recibe logits).

**Optimizadores:**
- **SGD**: el descenso de gradiente clásico (puede añadir `momentum`).
- **Adam / AdamW**: adapta la tasa de aprendizaje por parámetro; un excelente punto de partida (`lr=1e-3`).

In [ ]:
logits = torch.tensor([[2.0, 0.5, -1.0],
                       [0.1, 0.2, 3.0]])
etiquetas = torch.tensor([0, 2])

loss_fn = nn.CrossEntropyLoss()
print("Loss con etiquetas correctas  :", loss_fn(logits, etiquetas).item())
print("Loss con etiquetas incorrectas:",
      loss_fn(logits, torch.tensor([1, 0])).item())

print("\nSoftmax de los logits:\n", torch.softmax(logits, dim=1))

# Un optimizador recibe los parámetros que debe actualizar
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)
print("\nOptimizador:", type(optimizador).__name__)

#### 🔍 Qué hace este código

- `logits` son las **puntuaciones crudas** que "entregaría" una red para 2 ejemplos y 3 clases. Cada fila es un ejemplo. En la fila 1 la clase 0 tiene la puntuación más alta (2.0); en la fila 2 es la clase 2 (3.0).
- `etiquetas = [0, 2]` indica la **clase correcta** de cada ejemplo, como enteros (`int64`). Coinciden con las clases que el modelo prefiere, así que el error es pequeño: la pérdida sale **≈ 0.18**.
- Con `[1, 0]` (clases equivocadas) la pérdida sube a **≈ 2.38**: la función de pérdida penaliza más cuanto más confiado está el modelo en una respuesta incorrecta.
- `torch.softmax(logits, dim=1)` convierte cada fila de puntuaciones en **probabilidades que suman 1**. Es lo que `CrossEntropyLoss` hace por dentro (por eso **no** se aplica softmax a mano antes de la pérdida). `dim=1` indica que se normaliza a lo largo de las clases.
- `torch.optim.Adam(modelo.parameters(), lr=1e-3)` crea un optimizador al que se le **entregan los parámetros que debe actualizar** y la tasa de aprendizaje. Más adelante llamaremos a `optimizador.step()` para actualizarlos.

---
# 🔟 Regresión lineal con `torch.nn`

Repitamos el problema de la sección 8, pero ahora con las herramientas "oficiales". Mira cuánto se simplifica el código, y fíjate en las **5 líneas del ciclo de entrenamiento**:

```python
pred = model(x)                 # 1. forward
loss = loss_fn(pred, y)         # 2. calcular pérdida
optimizer.zero_grad()           # 3. limpiar gradientes anteriores
loss.backward()                 # 4. backward (calcular gradientes)
optimizer.step()                # 5. actualizar parámetros
```

In [ ]:
torch.manual_seed(42)

# Datos con forma (N, 1): nn.Linear espera (batch, características)
X = torch.linspace(-3, 3, 100).unsqueeze(1)
Y = 3 * X + 2 + torch.randn(100, 1) * 0.5

modelo = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizador = torch.optim.SGD(modelo.parameters(), lr=0.05)

for epoca in range(100):
    pred = modelo(X)                # 1. forward
    loss = loss_fn(pred, Y)         # 2. pérdida
    optimizador.zero_grad()         # 3. limpiar gradientes
    loss.backward()                 # 4. backward
    optimizador.step()              # 5. actualizar

    if epoca % 25 == 0:
        print(f"Época {epoca:3d} | loss = {loss.item():.4f}")

print(f"\nw = {modelo.weight.item():.3f} (real 3) | "
      f"b = {modelo.bias.item():.3f} (real 2)")

#### 🔍 Qué hace este código

- **Datos:** `torch.linspace(-3, 3, 100).unsqueeze(1)` genera los 100 puntos y los convierte en una **columna** `(100, 1)`. Esto es necesario porque `nn.Linear` espera tensores de forma `(ejemplos, características)`; aquí hay 1 característica. `Y` se construye igual que antes ($3X + 2$ + ruido).
- **Modelo:** `nn.Linear(1, 1)` es exactamente $\hat{y} = w x + b$, con un único peso y un único sesgo (reemplaza nuestros `w` y `b` manuales).
- **Pérdida:** `nn.MSELoss()` calcula el mismo error cuadrático medio que escribimos a mano.
- **Optimizador:** `SGD(modelo.parameters(), lr=0.05)` hace por nosotros las actualizaciones `w -= lr * w.grad`.
- **Ciclo:** las 5 líneas comentadas (forward, pérdida, `zero_grad`, `backward`, `step`) reemplazan todo el código manual de la sección 8. `optimizador.step()` actualiza todos los parámetros usando los gradientes ya calculados.
- Al final leemos `modelo.weight` y `modelo.bias`: deberían ser cercanos a 3 y 2, igual que con la versión manual.

> 🧠 **Idea clave:** `optimizer.step()` hace por ti lo que antes hacíamos a mano con `w -= lr * w.grad`. Con **Adam** o cualquier otro optimizador, el ciclo es idéntico; solo cambia la línea donde lo creas.

> ✏️ **Ejercicio 10.1** — (a) Compara `SGD` con `Adam(lr=0.1)` en este mismo problema lineal: ¿cuál reduce más rápido la pérdida? (b) Entrena un modelo no lineal `nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1))` con datos $y = x^2$.
>
> *Intenta resolverlo tú primero; la solución está en la celda de código que sigue.*

In [ ]:
# Solución
def entrenar_regresion(modelo, optimizador, X, Y, n_epocas):
    """Entrena un modelo de regresión con pérdida MSE.

    Args:
        modelo: instancia de ``nn.Module``.
        optimizador: optimizador ya creado con los parámetros del modelo.
        X: tensor de entradas con forma (N, características).
        Y: tensor de objetivos con forma (N, salidas).
        n_epocas: número de épocas de entrenamiento.

    Returns:
        Lista con la pérdida de cada época.
    """
    loss_fn = nn.MSELoss()
    perdidas = []
    for _ in range(n_epocas):
        loss = loss_fn(modelo(X), Y)
        optimizador.zero_grad()
        loss.backward()
        optimizador.step()
        perdidas.append(loss.item())
    return perdidas


# (a) SGD vs Adam en el problema lineal
resultados = {}
for nombre in ["SGD (lr=0.05)", "Adam (lr=0.1)"]:
    torch.manual_seed(42)
    m = nn.Linear(1, 1)
    if nombre.startswith("SGD"):
        opt = torch.optim.SGD(m.parameters(), lr=0.05)
    else:
        opt = torch.optim.Adam(m.parameters(), lr=0.1)
    resultados[nombre] = entrenar_regresion(m, opt, X, Y, n_epocas=100)

for nombre, perdidas in resultados.items():
    print(f"{nombre:14s} -> loss en épocas 0/10/30/99: "
          f"{perdidas[0]:.2f} / {perdidas[10]:.2f} / "
          f"{perdidas[30]:.2f} / {perdidas[99]:.2f}")

# (b) Modelo no lineal para y = x²
torch.manual_seed(0)
X2 = torch.linspace(-2, 2, 100).unsqueeze(1)
Y2 = X2 ** 2

modelo_nl = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1))
opt_nl = torch.optim.Adam(modelo_nl.parameters(), lr=0.01)
perdidas_nl = entrenar_regresion(modelo_nl, opt_nl, X2, Y2, n_epocas=500)
print(f"\nModelo no lineal: loss inicial {perdidas_nl[0]:.3f} -> "
      f"final {perdidas_nl[-1]:.5f}")

with torch.no_grad():
    pred_nl = modelo_nl(X2)

plt.figure(figsize=(5, 4))
plt.plot(X2.numpy(), Y2.numpy(), label="y = x²")
plt.plot(X2.numpy(), pred_nl.numpy(), "--", label="modelo")
plt.legend()
plt.title("Ajuste no lineal con una red pequeña")
plt.show()

#### 🔍 Qué hace este código

- **`entrenar_regresion`** empaqueta el ciclo de 5 pasos en una función reutilizable (con su docstring) que recibe el modelo, el optimizador y los datos, y devuelve el historial de pérdidas.
- **Parte (a):** entrenamos el mismo modelo lineal dos veces (con la misma semilla, para partir con los mismos pesos): una con SGD y otra con Adam, y comparamos la pérdida en las épocas 0, 10, 30 y 99. **Observación importante:** en un problema tan simple y con una tasa de aprendizaje bien elegida, **SGD puede igualar o incluso superar a Adam**. La ventaja de Adam aparece en problemas más complejos y en que es **mucho menos sensible a la tasa de aprendizaje elegida**, por eso es el optimizador por defecto de muchos proyectos.
- **Parte (b):** una recta jamás podría ajustar una parábola. Con una **capa oculta de 16 neuronas + ReLU** el modelo sí puede (cada ReLU aporta un "quiebre" y la combinación aproxima la curva). Al graficar, la línea punteada debería seguir de cerca a la parábola (puede notarse que está formada por tramos rectos, por la naturaleza de ReLU).
- `with torch.no_grad():` para predecir sin construir grafo, y `.numpy()` para graficar.

---
# 1️⃣1️⃣ `Dataset` y `DataLoader`

En la vida real, tus datos no caben de una sola vez en memoria (o en la GPU), y además entrenar con **lotes pequeños (*mini-batches*)** funciona mejor que con todo el conjunto junto. PyTorch resuelve esto con dos clases:

- **`Dataset`**: sabe **cuántos** ejemplos hay (`__len__`) y cómo entregar **el ejemplo i** (`__getitem__`).
- **`DataLoader`**: toma un Dataset y lo entrega **por lotes**, opcionalmente **barajado** (`shuffle=True`) y con **carga en paralelo** (`num_workers`).

## Vocabulario clave
| Término | Significado |
|---|---|
| **Época (*epoch*)** | Una pasada completa por **todos** los datos de entrenamiento |
| **Batch / lote** | Subconjunto de ejemplos procesados juntos en un paso |
| **Iteración (*step*)** | Una actualización de parámetros (= un batch) |

Ejemplo: 1000 datos con `batch_size=100` → **10 iteraciones por época**.

In [ ]:
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split

# Forma rápida: TensorDataset (cuando todo ya está en tensores)
X = torch.randn(1000, 5)
y = torch.randint(0, 2, (1000,))
ds = TensorDataset(X, y)
print("Tamaño del dataset:", len(ds))
ejemplo_x, ejemplo_y = ds[0]
print("Ejemplo 0 -> x:", tuple(ejemplo_x.shape), "| y:", ejemplo_y.item())

loader = DataLoader(ds, batch_size=100, shuffle=True)
print("Número de lotes:", len(loader))

xb, yb = next(iter(loader))
print("Un lote -> X:", tuple(xb.shape), "| y:", tuple(yb.shape))

# Recorrer el loader: así se usa dentro de un entrenamiento
for i, (xb, yb) in enumerate(loader):
    if i < 2:
        print(f"  lote {i}: X {tuple(xb.shape)}, y {tuple(yb.shape)}")
print(f"  ... (en total {i + 1} lotes por época)")

# Si el tamaño no es divisible por batch_size, el último lote es menor
tamanos = [len(yb) for _, yb in DataLoader(ds, batch_size=300)]
print("\nTamaños de lote con batch_size=300:", tamanos)

#### 🔍 Qué hace este código

- `X` son 1000 ejemplos con 5 características y `y` son 1000 etiquetas de clase (0 o 1).
- **`TensorDataset(X, y)`** une ambos tensores en un `Dataset`: `len(ds)` da 1000 y `ds[0]` entrega el **par (características, etiqueta)** del primer ejemplo.
- **`DataLoader(ds, batch_size=100, shuffle=True)`** crea un objeto iterable que entrega **lotes de 100 ejemplos**, **en orden aleatorio** distinto en cada época. `len(loader)` es el número de lotes: `1000 / 100 = 10`.
- `next(iter(loader))` obtiene **solo el primer lote** (útil para inspeccionar formas): `X` de `(100, 5)` e `y` de `(100,)`.
- El `for i, (xb, yb) in enumerate(loader)` es **cómo se usa el loader en un entrenamiento real**: cada vuelta entrega un lote. `enumerate` añade el contador `i`. Imprimimos solo los dos primeros lotes y, al final, el total (10).
- Con `batch_size=300`, 1000 ejemplos dan lotes de `[300, 300, 300, 100]`: **el último lote es más pequeño**. Si no lo quieres, `drop_last=True` lo descarta.

In [ ]:
class MiDataset(Dataset):
    """Dataset personalizado que envuelve tensores de entradas y etiquetas.

    Es el molde para cualquier dataset propio: aquí podrías leer
    archivos del disco, aplicar transformaciones, etc.

    Args:
        X: tensor con las entradas (una fila por ejemplo).
        y: tensor con las etiquetas.
    """

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        """Devuelve el número total de ejemplos."""
        return len(self.X)

    def __getitem__(self, idx):
        """Devuelve el par (entrada, etiqueta) del ejemplo ``idx``."""
        return self.X[idx], self.y[idx]


ds2 = MiDataset(X, y)
print("MiDataset tiene", len(ds2), "ejemplos")

# Dividir en entrenamiento y validación
generador = torch.Generator().manual_seed(0)
train_ds, val_ds = random_split(ds2, [800, 200], generator=generador)
print("Entrenamiento:", len(train_ds), "| Validación:", len(val_ds))

#### 🔍 Qué hace este código

- **`MiDataset`** es la plantilla de un `Dataset` propio. Solo necesita tres métodos:
  - `__init__`: guarda los datos (aquí, dos tensores).
  - `__len__`: dice **cuántos ejemplos** hay (lo usa `len(ds)` y el `DataLoader`).
  - `__getitem__(idx)`: devuelve **el ejemplo número `idx`**. Con datos reales (imágenes, texto), aquí abrirías el archivo y aplicarías transformaciones. El `DataLoader` llamará a este método muchas veces y apilará los resultados en lotes.
- **`random_split(ds2, [800, 200], generator=...)`** divide el dataset al azar en dos subconjuntos de 800 y 200 ejemplos. El `Generator` con semilla hace que la división sea **reproducible**.

> 💡 Para datos de **entrenamiento** usa `shuffle=True` (para que el orden no sesgue el aprendizaje). Para **validación/test** usa `shuffle=False`.

---
# 1️⃣2️⃣ Clasificación con una red multicapa (MLP)

Ahora un problema de **clasificación no lineal**: tres espirales entrelazadas en 2D. Una línea recta no puede separarlas, pero una red con capas ocultas y activaciones sí. Vamos a armar el **flujo completo y profesional**: datos → Dataset/DataLoader → modelo → entrenamiento → validación → visualización.

In [ ]:
torch.manual_seed(0)


def hacer_espirales(n_por_clase=300, n_clases=3, ruido=0.15):
    """Genera un dataset 2D de espirales entrelazadas, una por clase.

    Args:
        n_por_clase: número de puntos de cada espiral.
        n_clases: número de espirales (clases).
        ruido: desviación estándar del ruido añadido al ángulo.

    Returns:
        Tupla ``(X, y)``: ``X`` de forma (N, 2) con las coordenadas
        (``float32``) e ``y`` de forma (N,) con la clase (``int64``).
    """
    puntos, etiquetas = [], []
    for clase in range(n_clases):
        radio = torch.linspace(0.05, 1, n_por_clase)
        angulo = torch.linspace(clase * 4, (clase + 1) * 4, n_por_clase)
        angulo = angulo + torch.randn(n_por_clase) * ruido
        coordenadas = torch.stack([radio * torch.sin(angulo),
                                   radio * torch.cos(angulo)], dim=1)
        puntos.append(coordenadas)
        etiquetas.append(torch.full((n_por_clase,), clase,
                                    dtype=torch.long))
    return torch.cat(puntos), torch.cat(etiquetas)


X, y = hacer_espirales()
print("X:", tuple(X.shape), X.dtype, "| y:", tuple(y.shape), y.dtype)

plt.figure(figsize=(5, 5))
plt.scatter(X[:, 0].numpy(), X[:, 1].numpy(), c=y.numpy(), s=10,
            cmap="brg")
plt.title("Tres espirales (3 clases)")
plt.show()

#### 🔍 Qué hace este código

**La función `hacer_espirales`** construye un dataset de juguete:
1. Para cada clase se crea una espiral: el `radio` crece de 0.05 a 1 y el `angulo` recorre un intervalo distinto por clase (`[0, 4]`, `[4, 8]`, `[8, 12]` radianes), más un poco de ruido. Eso hace que las tres espirales se enrosquen sin superponerse del todo.
2. `radio * sin(angulo)` y `radio * cos(angulo)` son las coordenadas polares convertidas a (x, y). `torch.stack(..., dim=1)` las junta en una matriz de `(n_por_clase, 2)`: una fila por punto.
3. `torch.full((n,), clase, dtype=torch.long)` crea el vector de etiquetas (todas iguales a la clase actual). **`long` (int64) es obligatorio** para `CrossEntropyLoss`.
4. `torch.cat` pega las tres espirales en un solo tensor.

**Resultado:** `X` de forma `(900, 2)` (`float32`) e `y` de forma `(900,)` (`int64`), con 300 puntos por clase. El `scatter` los dibuja coloreados por clase (`c=y`) y se ven como tres brazos de espiral entrelazados. **Una recta no puede separar estas clases**: necesitamos una red con no linealidades.

In [ ]:
# Dividir en entrenamiento (80%) y validación (20%)
ds = TensorDataset(X, y)
n_train = int(0.8 * len(ds))
train_ds, val_ds = random_split(
    ds, [n_train, len(ds) - n_train],
    generator=torch.Generator().manual_seed(0))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)
print("Entrenamiento:", len(train_ds), "| Validación:", len(val_ds))
print("Lotes de entrenamiento:", len(train_loader))

#### 🔍 Qué hace este código

- `TensorDataset(X, y)` une entradas y etiquetas; `int(0.8 * len(ds))` calcula cuántos ejemplos (720) irán a entrenamiento.
- `random_split` reparte **al azar** el dataset en 720 ejemplos de entrenamiento y 180 de validación (la semilla del generador hace que sea reproducible).
- `train_loader` entrega lotes de 64 y **baraja** los datos en cada época (12 lotes por época, porque `720 / 64 = 11.25`, así que el último es más pequeño). `val_loader` entrega lotes grandes (256) y **sin barajar**: al evaluar no hace falta, y como no se calculan gradientes cabe más.

In [ ]:
def entrenar_una_epoca(modelo, loader, loss_fn, optimizador, device):
    """Entrena el modelo durante una época completa.

    Args:
        modelo: red neuronal (``nn.Module``) ya movida a ``device``.
        loader: ``DataLoader`` con los datos de entrenamiento.
        loss_fn: función de pérdida (por ejemplo ``CrossEntropyLoss``).
        optimizador: optimizador asociado a los parámetros del modelo.
        device: dispositivo donde se hacen los cálculos.

    Returns:
        Tupla ``(pérdida_promedio, exactitud)`` de la época.
    """
    modelo.train()
    perdida_total, aciertos, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        logits = modelo(xb)                # 1. forward
        loss = loss_fn(logits, yb)         # 2. pérdida
        optimizador.zero_grad()            # 3. limpiar gradientes
        loss.backward()                    # 4. backward
        optimizador.step()                 # 5. actualizar

        perdida_total += loss.item() * xb.size(0)
        aciertos += (logits.argmax(dim=1) == yb).sum().item()
        n += xb.size(0)
    return perdida_total / n, aciertos / n


@torch.no_grad()
def evaluar(modelo, loader, loss_fn, device):
    """Evalúa el modelo sin modificar sus parámetros.

    Args:
        modelo: red neuronal (``nn.Module``) ya movida a ``device``.
        loader: ``DataLoader`` con los datos a evaluar.
        loss_fn: función de pérdida.
        device: dispositivo donde se hacen los cálculos.

    Returns:
        Tupla ``(pérdida_promedio, exactitud)``.
    """
    modelo.eval()
    perdida_total, aciertos, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = modelo(xb)
        perdida_total += loss_fn(logits, yb).item() * xb.size(0)
        aciertos += (logits.argmax(dim=1) == yb).sum().item()
        n += xb.size(0)
    return perdida_total / n, aciertos / n

#### 🔍 Qué hace este código

Estas dos funciones se reutilizan durante todo el notebook.

**`entrenar_una_epoca`** (una pasada completa por los datos de entrenamiento):
- `modelo.train()` activa el **modo entrenamiento** (importa para capas como Dropout; ver sección 13).
- Por cada lote: se **mueve al dispositivo** (`.to(device)`), se hacen los 5 pasos del ciclo (forward, pérdida, `zero_grad`, backward, `step`).
- Para las métricas acumulamos:
  - la **pérdida**: `loss.item() * xb.size(0)` (pérdida del lote × tamaño del lote, para luego promediar correctamente aunque el último lote sea menor);
  - los **aciertos**: `logits.argmax(dim=1)` es la clase con mayor puntuación (la predicha); `== yb` compara con la real; `.sum().item()` cuenta cuántas coinciden.
- Devuelve la pérdida promedio y la **exactitud** (*accuracy*: fracción de aciertos).

**`evaluar`** hace lo mismo **sin entrenar**:
- El decorador `@torch.no_grad()` apaga el seguimiento de gradientes en toda la función (más rápido y con menos memoria).
- `modelo.eval()` activa el **modo evaluación**.
- No hay `backward` ni `step`: solo medimos.

In [ ]:
torch.manual_seed(0)

modelo = nn.Sequential(
    nn.Linear(2, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 3),
).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-2)

hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoca in range(1, 101):
    tl, ta = entrenar_una_epoca(modelo, train_loader, loss_fn,
                                optimizador, device)
    vl, va = evaluar(modelo, val_loader, loss_fn, device)
    hist["train_loss"].append(tl)
    hist["val_loss"].append(vl)
    hist["train_acc"].append(ta)
    hist["val_acc"].append(va)
    if epoca % 20 == 0 or epoca == 1:
        print(f"Época {epoca:3d} | train loss {tl:.3f} acc {ta:.3f} | "
              f"val loss {vl:.3f} acc {va:.3f}")

#### 🔍 Qué hace este código

- **Modelo:** una red `2 → 64 → ReLU → 64 → ReLU → 3`. Recibe las 2 coordenadas y entrega 3 puntuaciones (una por clase). `.to(device)` la mueve a la GPU si hay.
- **Pérdida y optimizador:** `CrossEntropyLoss` (clasificación multiclase) y `Adam` con `lr=0.01`.
- `hist` es un diccionario con listas vacías donde guardamos las métricas de cada época para graficarlas después.
- **Bucle de épocas (1 a 100):** en cada una llamamos a `entrenar_una_epoca` (aprende) y a `evaluar` sobre los datos de validación (mide cómo generaliza a ejemplos que **no** usó para aprender). Guardamos todo en `hist` e imprimimos en las épocas 1, 20, 40, ..., 100.

**Qué esperar:** en la época 1 la pérdida ronda `1.1` (≈ ln 3, que es lo que da elegir al azar entre 3 clases) y la exactitud ~0.33–0.5. A medida que avanzan las épocas, la pérdida baja y la exactitud sube, normalmente por encima de 0.9 tanto en entrenamiento como en validación.

In [ ]:
def graficar_historial(hist):
    """Grafica las curvas de pérdida y exactitud de entrenamiento.

    Args:
        hist: diccionario con las listas ``train_loss``, ``val_loss``,
            ``train_acc`` y ``val_acc``.
    """
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(hist["train_loss"], label="entrenamiento")
    ax[0].plot(hist["val_loss"], label="validación")
    ax[0].set_title("Pérdida")
    ax[0].set_xlabel("Época")
    ax[0].legend()
    ax[1].plot(hist["train_acc"], label="entrenamiento")
    ax[1].plot(hist["val_acc"], label="validación")
    ax[1].set_title("Exactitud (accuracy)")
    ax[1].set_xlabel("Época")
    ax[1].legend()
    plt.show()


def graficar_frontera(modelo, X, y, device):
    """Dibuja la región de decisión de un clasificador 2D.

    Args:
        modelo: clasificador ya entrenado (entrada de 2 características).
        X: tensor (N, 2) con los puntos a dibujar.
        y: tensor (N,) con las clases reales (para colorear).
        device: dispositivo donde está el modelo.
    """
    modelo.eval()
    xs = torch.linspace(X[:, 0].min().item() - 0.1,
                        X[:, 0].max().item() + 0.1, 300)
    ys = torch.linspace(X[:, 1].min().item() - 0.1,
                        X[:, 1].max().item() + 0.1, 300)
    gx, gy = torch.meshgrid(xs, ys, indexing="xy")
    grilla = torch.stack([gx.ravel(), gy.ravel()], dim=1).to(device)
    with torch.no_grad():
        pred = modelo(grilla).argmax(dim=1).cpu()
    plt.figure(figsize=(5, 5))
    plt.contourf(gx.numpy(), gy.numpy(), pred.reshape(gx.shape).numpy(),
                 alpha=0.3, cmap="brg")
    plt.scatter(X[:, 0].numpy(), X[:, 1].numpy(), c=y.numpy(), s=10,
                cmap="brg")
    plt.title("Frontera de decisión aprendida")
    plt.show()


graficar_historial(hist)
graficar_frontera(modelo, X, y, device)

#### 🔍 Qué hace este código

- **`graficar_historial`** dibuja, para pérdida y exactitud, una curva de entrenamiento y otra de validación. Es la herramienta principal para **diagnosticar el entrenamiento** (ver sección 13).
- **`graficar_frontera`** muestra **qué región del plano asigna el modelo a cada clase**:
  1. `torch.linspace` crea 300 valores para el eje x y 300 para el eje y, cubriendo el rango de los datos con un pequeño margen.
  2. `torch.meshgrid` genera **todas las combinaciones** (una grilla de 300×300 puntos) y `torch.stack([gx.ravel(), gy.ravel()], dim=1)` las convierte en una lista de 90.000 puntos `(x, y)`.
  3. Se la damos al modelo, y `argmax(dim=1)` entrega la **clase predicha para cada punto de la grilla**. Se pasa a `.cpu()` para poder graficar.
  4. `contourf` pinta las regiones de colores y `scatter` dibuja encima los datos reales.
- Si el modelo aprendió bien, las zonas de color deberían **seguir los brazos de las espirales** y casi todos los puntos caen sobre su color.

## Qué observar
- La **pérdida** debería bajar y la **exactitud** subir en entrenamiento y validación.
- Si la métrica de entrenamiento mejora mucho pero la de validación se estanca o empeora → **sobreajuste** (sección siguiente).
- Fíjate en `modelo.train()` / `modelo.eval()` y en `@torch.no_grad()`: son hábitos que debes incorporar siempre.

> ✏️ **Ejercicio 12.1** — Prueba cambiar el modelo: (a) quita las activaciones `ReLU` (¿qué pasa con la frontera?), (b) usa solo 4 neuronas ocultas, (c) usa `lr=1e-4`. Anota qué ocurre.
>
> *Intenta resolverlo tú primero; la solución está en la celda de código que sigue.*

In [ ]:
# Solución
def crear_mlp(n_ocultas=64, con_activacion=True):
    """Crea un MLP 2 -> n_ocultas -> n_ocultas -> 3 en ``device``.

    Args:
        n_ocultas: neuronas de cada capa oculta.
        con_activacion: si es False, omite las ReLU (modelo lineal).

    Returns:
        Un ``nn.Sequential`` listo para entrenar.
    """
    capas = [nn.Linear(2, n_ocultas)]
    if con_activacion:
        capas.append(nn.ReLU())
    capas.append(nn.Linear(n_ocultas, n_ocultas))
    if con_activacion:
        capas.append(nn.ReLU())
    capas.append(nn.Linear(n_ocultas, 3))
    return nn.Sequential(*capas).to(device)


def probar_configuracion(modelo, lr, n_epocas=100):
    """Entrena ``modelo`` y devuelve su exactitud final de validación."""
    optimizador = torch.optim.Adam(modelo.parameters(), lr=lr)
    for _ in range(n_epocas):
        entrenar_una_epoca(modelo, train_loader, loss_fn, optimizador,
                           device)
    return evaluar(modelo, val_loader, loss_fn, device)[1]


experimentos = [
    ("Base: 64 neuronas, ReLU, lr=1e-2", 64, True, 1e-2),
    ("(a) Sin activaciones", 64, False, 1e-2),
    ("(b) Solo 4 neuronas ocultas", 4, True, 1e-2),
    ("(c) lr = 1e-4", 64, True, 1e-4),
]

modelos_exp = {}
for nombre, n_ocultas, con_act, lr_exp in experimentos:
    torch.manual_seed(0)
    modelo_exp = crear_mlp(n_ocultas, con_act)
    exactitud = probar_configuracion(modelo_exp, lr_exp)
    modelos_exp[nombre] = modelo_exp
    print(f"{nombre:36s} -> exactitud de validación: {exactitud:.3f}")

graficar_frontera(modelos_exp["(a) Sin activaciones"], X, y, device)

#### 🔍 Qué hace este código

- `crear_mlp` construye el modelo con la configuración deseada (número de neuronas y presencia de ReLU) y lo mueve a `device`. `nn.Sequential(*capas)` desempaqueta la lista de capas como argumentos.
- `probar_configuracion` entrena 100 épocas y devuelve la exactitud de validación (`evaluar(...)[1]` toma el segundo valor de la tupla).
- El bucle prueba las cuatro configuraciones (con la misma semilla, para una comparación justa) e imprime su exactitud.

**Qué verás y por qué:**
- **Base:** exactitud alta (normalmente > 0.9).
- **(a) Sin ReLU:** al no haber no linealidades, 3 capas lineales seguidas equivalen a **una sola** transformación lineal. La frontera es de **rectas** (se ve en el gráfico) y no puede separar espirales: la exactitud es muy inferior.
- **(b) 4 neuronas:** el modelo tiene **poca capacidad** para una curva tan retorcida: la exactitud baja respecto de la base (subajuste).
- **(c) `lr = 1e-4`:** la tasa es tan pequeña que en 100 épocas el modelo **apenas avanza**: exactitud mucho menor (entrenamiento lento).

---
# 1️⃣3️⃣ Sobreajuste y regularización

## ¿Qué es el sobreajuste (*overfitting*)?
El modelo **memoriza** los datos de entrenamiento (incluido su ruido) en vez de aprender el patrón general, y por eso rinde mal con datos nuevos.

> 🎓 **Analogía:** un estudiante que memoriza las respuestas de los ejercicios de la guía, pero no sabe resolver uno nuevo en la prueba.

**Cómo detectarlo:** la pérdida de *entrenamiento* sigue bajando, pero la de *validación* se estanca o sube.

## Divisiones de datos
| Conjunto | Para qué sirve |
|---|---|
| **Entrenamiento (train)** | Ajustar los parámetros del modelo |
| **Validación (val)** | Ajustar decisiones (hiperparámetros, cuándo parar) |
| **Prueba (test)** | Medida **final** e imparcial; se usa una sola vez |

## Técnicas de regularización
- **Más datos / aumento de datos** (*data augmentation*).
- **Dropout** (`nn.Dropout(p)`): durante el entrenamiento apaga neuronas al azar con probabilidad `p`, forzando a la red a no depender de una sola.
- **Weight decay**: penaliza pesos grandes (`optim.AdamW(..., weight_decay=0.01)`).
- **Early stopping**: detener el entrenamiento cuando la validación deja de mejorar.
- **Modelo más pequeño**.

## `model.train()` vs `model.eval()`
Algunas capas se comportan **distinto** al entrenar y al evaluar:
- `Dropout`: activo en `train()`, **desactivado** en `eval()`.
- `BatchNorm`: usa estadísticas del lote en `train()`, y estadísticas acumuladas en `eval()`.

Olvidar `model.eval()` al evaluar es un error clásico que da resultados erráticos.

## 13.1 Ver el sobreajuste con un ejemplo
Ajustaremos una curva $y = \sin(x)$ con solo **30 puntos ruidosos** usando una red grande. Compararemos el error en los datos de entrenamiento con el error contra la **función verdadera** (sin ruido) en puntos nuevos.

In [ ]:
import copy

torch.manual_seed(1)

# Pocos datos ruidosos de y = sin(x)
x_tr = torch.rand(30, 1) * 6 - 3
y_tr = torch.sin(x_tr) + torch.randn(30, 1) * 0.3

# Datos de prueba: la función verdadera, sin ruido
x_te = torch.linspace(-3, 3, 200).unsqueeze(1)
y_te = torch.sin(x_te)


def entrenar_regresor(modelo, x, y, n_epocas=2000, lr=1e-2,
                      weight_decay=0.0):
    """Entrena un regresor con todo el conjunto en cada paso (sin lotes).

    Args:
        modelo: red a entrenar.
        x: entradas de entrenamiento, forma (N, 1).
        y: objetivos de entrenamiento, forma (N, 1).
        n_epocas: número de pasos de entrenamiento.
        lr: tasa de aprendizaje.
        weight_decay: penalización de pesos grandes (regularización).

    Returns:
        El mismo modelo, ya entrenado y en modo evaluación.
    """
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=lr,
                                    weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    modelo.train()
    for _ in range(n_epocas):
        loss = loss_fn(modelo(x), y)
        optimizador.zero_grad()
        loss.backward()
        optimizador.step()
    modelo.eval()
    return modelo


@torch.no_grad()
def mse(modelo, x, y):
    """Calcula el error cuadrático medio del modelo sobre (x, y)."""
    modelo.eval()
    return nn.functional.mse_loss(modelo(x), y).item()


# Modelo grande sin regularizar
modelo_grande = nn.Sequential(
    nn.Linear(1, 128), nn.ReLU(),
    nn.Linear(128, 128), nn.ReLU(),
    nn.Linear(128, 1),
)
entrenar_regresor(modelo_grande, x_tr, y_tr)

# Mismo tamaño, con Dropout y weight decay
modelo_reg2 = nn.Sequential(
    nn.Linear(1, 128), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(128, 128), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(128, 1),
)
entrenar_regresor(modelo_reg2, x_tr, y_tr, weight_decay=0.1)

for nombre, m in [("Sin regularizar", modelo_grande),
                  ("Con dropout + weight decay", modelo_reg2)]:
    print(f"{nombre:28s} | error en train: {mse(m, x_tr, y_tr):.4f} | "
          f"error vs función real: {mse(m, x_te, y_te):.4f}")

with torch.no_grad():
    pred_grande = modelo_grande(x_te).numpy()
    pred_reg = modelo_reg2(x_te).numpy()

plt.figure(figsize=(8, 4))
plt.scatter(x_tr.numpy(), y_tr.numpy(), color="black", label="datos (ruido)")
plt.plot(x_te.numpy(), y_te.numpy(), "g", label="función real")
plt.plot(x_te.numpy(), pred_grande, "r", label="sin regularizar")
plt.plot(x_te.numpy(), pred_reg, "b", label="regularizado")
plt.legend()
plt.show()

#### 🔍 Qué hace este código

- **Datos:** 30 puntos de `sin(x)` con ruido (`x_tr`, `y_tr`) y 200 puntos de prueba de la función **verdadera** (`x_te`, `y_te`). Medir contra la función real nos dice cuánto aprendió el *patrón* y no el ruido.
- **`entrenar_regresor`** entrena con todos los datos en cada paso (son pocos). Usa `AdamW`, que incluye `weight_decay` (penalización de pesos grandes). `mse` calcula el error cuadrático medio de un modelo sobre unos datos.
- **Dos modelos del mismo tamaño** (128 neuronas por capa, mucho más de lo que 30 puntos necesitan): el primero sin regularizar, el segundo con **Dropout (0.2)** y **weight decay**.
- Imprimimos para cada uno el error en los datos de entrenamiento y el error contra la función real.

**Qué esperar:** el modelo sin regularizar suele conseguir un **error de entrenamiento muy pequeño** (memorizó el ruido) pero un error contra la función real **mayor**. En el gráfico, su curva roja se ve **ondulada**, forzada a pasar por los puntos ruidosos. El modelo regularizado tiende a una curva más **suave**, más cercana a la verde. (Los números exactos varían; la diferencia entre "error en train" y "error real" es lo que delata el sobreajuste.)

In [ ]:
# Demostración de Dropout
drop = nn.Dropout(p=0.5)
x = torch.ones(10)

drop.train()
print("modo train:", drop(x))

drop.eval()
print("modo eval :", drop(x))

#### 🔍 Qué hace este código

- `nn.Dropout(p=0.5)` apaga cada elemento con probabilidad 50 % durante el entrenamiento.
- En **modo `train()`**, aproximadamente la mitad de los valores se vuelven `0` y **los que sobreviven se multiplican por `1 / (1 - p) = 2`** (para que la suma total esperada se mantenga). Verás una mezcla de `0.` y `2.` distinta cada vez que ejecutes la celda.
- En **modo `eval()`**, no hace nada: la salida es `[1., 1., ..., 1.]`. Por eso **olvidar `model.eval()` al evaluar es un error grave**: el modelo seguiría apagando neuronas al azar y los resultados saldrían erráticos.

## 13.2 Early stopping (parada temprana)
Volvemos al problema de las espirales: guardamos los mejores pesos según la pérdida de validación y detenemos el entrenamiento cuando deja de mejorar durante cierto número de épocas (la *paciencia*).

In [ ]:
torch.manual_seed(0)

modelo_reg = nn.Sequential(
    nn.Linear(2, 64), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(64, 3),
).to(device)

optimizador = torch.optim.AdamW(modelo_reg.parameters(), lr=1e-2,
                                weight_decay=1e-2)

mejor_val = float("inf")
paciencia = 10
sin_mejora = 0
mejor_estado = None

for epoca in range(1, 201):
    entrenar_una_epoca(modelo_reg, train_loader, loss_fn, optimizador,
                       device)
    val_loss, val_acc = evaluar(modelo_reg, val_loader, loss_fn, device)

    if val_loss < mejor_val - 1e-4:
        mejor_val = val_loss
        sin_mejora = 0
        mejor_estado = copy.deepcopy(modelo_reg.state_dict())
    else:
        sin_mejora += 1
        if sin_mejora >= paciencia:
            print(f"Early stopping en la época {epoca} "
                  f"(mejor val loss = {mejor_val:.3f})")
            break

modelo_reg.load_state_dict(mejor_estado)
print("Exactitud de validación final:",
      round(evaluar(modelo_reg, val_loader, loss_fn, device)[1], 3))

#### 🔍 Qué hace este código

- El modelo usa **Dropout** y el optimizador **AdamW con weight decay**: dos regularizadores.
- Variables de control:
  - `mejor_val` guarda la mejor pérdida de validación vista (parte en infinito).
  - `paciencia = 10`: cuántas épocas seguidas toleramos sin mejora.
  - `sin_mejora` cuenta las épocas consecutivas sin mejorar.
  - `mejor_estado` guardará una copia de los mejores pesos.
- En cada época: se entrena, se evalúa en validación y se decide:
  - **Si mejora** (con un margen de `1e-4`): actualizamos `mejor_val`, reiniciamos el contador y guardamos una copia de los pesos con `copy.deepcopy(modelo.state_dict())`. Hace falta la **copia profunda** porque `state_dict()` devuelve referencias a los tensores vivos del modelo, que seguirían cambiando.
  - **Si no mejora**: sumamos 1 al contador; si llega a la paciencia, `break` detiene el bucle.
- Al terminar, `load_state_dict(mejor_estado)` **restaura los mejores pesos** (no los del último paso, que pueden ser peores) y evaluamos.
- Puede que el entrenamiento llegue a las 200 épocas sin activar el *early stopping*; en ese caso simplemente no se imprime el aviso.

---
# 1️⃣4️⃣ Guardar y cargar modelos

Entrenar cuesta tiempo; no quieres repetirlo cada vez. La forma recomendada es guardar el **`state_dict`**: un diccionario que mapea el nombre de cada parámetro a su tensor.

- **Guardar:** `torch.save(model.state_dict(), "archivo.pt")`
- **Cargar:** crear **el mismo modelo** (misma arquitectura) y luego `model.load_state_dict(torch.load("archivo.pt"))`.

Para poder **reanudar** un entrenamiento, guarda también el estado del optimizador y la época (*checkpoint*).

In [ ]:
# Qué contiene un state_dict
print("Claves del state_dict:", list(modelo_reg.state_dict().keys()))

# Guardar solo los pesos
torch.save(modelo_reg.state_dict(), "modelo_espirales.pt")

# Cargar en un modelo NUEVO con la misma arquitectura
modelo_nuevo = nn.Sequential(
    nn.Linear(2, 64), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(64, 3),
).to(device)

estado = torch.load("modelo_espirales.pt", map_location=device,
                    weights_only=True)
modelo_nuevo.load_state_dict(estado)
modelo_nuevo.eval()

print("Exactitud tras cargar:",
      round(evaluar(modelo_nuevo, val_loader, loss_fn, device)[1], 3))

# Checkpoint completo (para reanudar un entrenamiento)
checkpoint = {
    "epoca": epoca,
    "modelo": modelo_reg.state_dict(),
    "optimizador": optimizador.state_dict(),
}
torch.save(checkpoint, "checkpoint.pt")

recuperado = torch.load("checkpoint.pt", map_location=device,
                        weights_only=True)
print("Checkpoint recuperado; época guardada:", recuperado["epoca"])

#### 🔍 Qué hace este código

- **`state_dict()`** es un diccionario `nombre → tensor`. Sus claves son `0.weight`, `0.bias`, `3.weight`, `3.bias`, `6.weight`, `6.bias`: los números son las **posiciones de las capas con parámetros dentro del `Sequential`** (las posiciones 1, 2, 4 y 5 son ReLU y Dropout, que no tienen parámetros).
- **`torch.save(..., "modelo_espirales.pt")`** escribe los pesos en un archivo (en la carpeta actual del notebook). Se guardan **solo los números**, no el código del modelo.
- **Cargar:** como el archivo no contiene la arquitectura, hay que **volver a crear el mismo modelo** y luego volcar los pesos con `load_state_dict`.
  - `map_location=device` carga los tensores directamente en el dispositivo deseado (permite abrir en CPU un modelo entrenado en GPU).
  - `weights_only=True` es una medida de **seguridad**: restringe la carga a datos tensoriales y evita ejecutar código arbitrario que pudiera venir en un archivo ajeno (los archivos de `torch.save` usan `pickle`, que puede ser peligroso).
- **`modelo_nuevo.eval()`** es imprescindible para predecir. Luego comprobamos que su exactitud coincide con la del modelo original.
- **Checkpoint:** guardamos en un diccionario la época, los pesos del modelo **y el estado del optimizador** (Adam guarda internamente promedios de gradientes). Con eso se puede **reanudar el entrenamiento exactamente donde quedó**. Lo cargamos de nuevo y mostramos la época guardada.

---
# 1️⃣5️⃣ Redes convolucionales (CNN) con FashionMNIST

Ahora un ejemplo real de **visión por computador**. **FashionMNIST** contiene 70.000 imágenes en escala de grises de 28×28 píxeles de 10 tipos de prendas (camiseta, pantalón, zapatilla...). *(La primera vez se descarga automáticamente: requiere internet.)*

## ¿Por qué una CNN y no un MLP?
Una imagen tiene **estructura espacial**: los píxeles vecinos se relacionan. Las capas convolucionales:
- Usan **filtros pequeños** (por ejemplo 3×3) que se deslizan por la imagen buscando patrones (bordes, texturas, formas).
- **Comparten pesos** a lo largo de la imagen → muchos menos parámetros que un MLP.
- Detectan un patrón **sin importar en qué posición** aparezca.

## Piezas
| Capa | Qué hace |
|---|---|
| `nn.Conv2d(c_in, c_out, k)` | Aplica `c_out` filtros de tamaño `k×k` |
| `nn.MaxPool2d(2)` | Reduce a la mitad el alto y ancho quedándose con el máximo de cada bloque 2×2 |
| `nn.Flatten()` | Aplana el mapa de características a un vector para la capa lineal |

**Formato de imágenes en PyTorch:** `(batch, canales, alto, ancho)` → **NCHW**.

In [ ]:
from torchvision import datasets, transforms

transformacion = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])

train_data = datasets.FashionMNIST(root="data", train=True,
                                   download=True,
                                   transform=transformacion)
test_data = datasets.FashionMNIST(root="data", train=False,
                                  download=True,
                                  transform=transformacion)

clases = train_data.classes
print("Clases:", clases)
print("Tamaño train:", len(train_data), "| test:", len(test_data))

img, etiqueta = train_data[0]
print("Una imagen:", tuple(img.shape), "| etiqueta:", etiqueta,
      f"({clases[etiqueta]})")

#### 🔍 Qué hace este código

- `transforms.Compose([...])` encadena transformaciones que se aplican a **cada imagen** al leerla:
  - `ToTensor()`: convierte la imagen (formato PIL) en un tensor `(canales, alto, ancho)` con los píxeles escalados de 0–255 a **0–1**.
  - `Normalize((0.2860,), (0.3530,))`: resta la media y divide por la desviación estándar de FashionMNIST (`(x - media) / desv`). Los datos centrados en 0 facilitan el entrenamiento.
- `datasets.FashionMNIST(root="data", train=True, download=True, ...)`: descarga el dataset a la carpeta `data` (solo la primera vez) y lo expone como un `Dataset`. `train=True` selecciona los 60.000 ejemplos de entrenamiento y `train=False` los 10.000 de prueba.
- `train_data.classes` es la lista con los nombres de las 10 clases (`T-shirt/top`, `Trouser`, `Pullover`, ...).
- `train_data[0]` devuelve el par (imagen, etiqueta): la imagen tiene forma `(1, 28, 28)` (**1 canal** porque es en blanco y negro) y la etiqueta es un entero que usamos como índice en `clases`.

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for ax, i in zip(axes.ravel(), range(12)):
    imagen, etiqueta = train_data[i]
    ax.imshow(imagen.squeeze(0).numpy(), cmap="gray")
    ax.set_title(clases[etiqueta], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

#### 🔍 Qué hace este código

- `plt.subplots(2, 6)` crea una grilla de 2 filas × 6 columnas de gráficos; `axes.ravel()` la "aplana" en una lista de 12 ejes para recorrerlos con un solo `for`.
- `zip(axes.ravel(), range(12))` empareja cada eje con el índice de una imagen.
- `imagen.squeeze(0)` **elimina la dimensión de canal** (de `(1, 28, 28)` a `(28, 28)`), porque `imshow` espera una matriz 2D para imágenes en gris.
- Cada imagen se muestra con su nombre de clase como título y sin ejes (`axis("off")`). Verás prendas de vestir en baja resolución.

In [ ]:
train_loader_fm = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader_fm = DataLoader(test_data, batch_size=512, shuffle=False)


class SimpleCNN(nn.Module):
    """Red convolucional pequeña para imágenes de 1 canal y 28x28.

    Dos bloques (convolución + ReLU + max-pooling) extraen
    características; luego un clasificador lineal con dropout decide
    la clase.

    Args:
        n_clases: número de clases de salida.
    """

    def __init__(self, n_clases=10):
        super().__init__()
        self.features = nn.Sequential(
            # (1, 28, 28) -> (16, 14, 14)
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # (16, 14, 14) -> (32, 7, 7)
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                    # (32, 7, 7) -> 1568
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_clases),
        )

    def forward(self, x):
        """Devuelve los logits para un lote de imágenes ``x``."""
        return self.classifier(self.features(x))


cnn = SimpleCNN().to(device)
print(cnn)
print("Parámetros:", sum(p.numel() for p in cnn.parameters()))

# Truco para depurar formas: pasar un lote falso por el modelo
lote_falso = torch.randn(8, 1, 28, 28, device=device)
print("Salida para 8 imágenes:", tuple(cnn(lote_falso).shape))

#### 🔍 Qué hace este código

- **DataLoaders:** lotes de 128 imágenes para entrenar (barajadas) y de 512 para probar. Les ponemos el sufijo `_fm` para no pisar los loaders de las espirales.
- **La arquitectura `SimpleCNN`** se divide en dos partes:
  1. **`features`** (extractor de características), con dos bloques `Conv2d → ReLU → MaxPool2d`:
     - `Conv2d(1, 16, kernel_size=3, padding=1)`: 16 filtros de 3×3 sobre la imagen de 1 canal. `padding=1` añade un borde de ceros para que el tamaño **no cambie** (28×28 → 28×28), y ahora hay 16 "mapas" (canales).
     - `MaxPool2d(2)`: reduce a la mitad el alto y el ancho (28 → 14), quedándose con el máximo de cada bloque 2×2.
     - Segundo bloque: 16 → 32 canales y 14 → 7. El tamaño final es **(32, 7, 7)**.
  2. **`classifier`**: `Flatten` aplana `(32, 7, 7)` en un vector de `32·7·7 = 1568` valores, que pasa por una capa lineal de 128 neuronas, ReLU, **Dropout** (regularización) y una capa final de 10 salidas (una por clase).
- `forward` encadena ambas partes.
- **Contar parámetros:** da ≈ **207 mil** (la mayoría, ~200 mil, están en la primera capa lineal del clasificador; las convoluciones usan muy pocos porque comparten pesos).
- **Truco de depuración:** pasar un lote falso `(8, 1, 28, 28)` por el modelo y revisar la forma de salida. Debe ser `(8, 10)`: 8 imágenes × 10 clases. Si hubiera un error de dimensiones, aparecería aquí, antes de perder tiempo entrenando.

In [ ]:
torch.manual_seed(0)
loss_fn = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(cnn.parameters(), lr=1e-3)

hist_cnn = {"train_loss": [], "val_loss": [], "train_acc": [],
            "val_acc": []}
n_epocas = 5   # en CPU puede tardar un poco; en GPU es muy rápido

for epoca in range(1, n_epocas + 1):
    tl, ta = entrenar_una_epoca(cnn, train_loader_fm, loss_fn,
                                optimizador, device)
    vl, va = evaluar(cnn, test_loader_fm, loss_fn, device)
    hist_cnn["train_loss"].append(tl)
    hist_cnn["val_loss"].append(vl)
    hist_cnn["train_acc"].append(ta)
    hist_cnn["val_acc"].append(va)
    print(f"Época {epoca} | train loss {tl:.3f} acc {ta:.3f} | "
          f"test loss {vl:.3f} acc {va:.3f}")

graficar_historial(hist_cnn)

#### 🔍 Qué hace este código

- Reutilizamos `entrenar_una_epoca`, `evaluar` y `graficar_historial` de la sección 12: **el mismo código sirve para un MLP y para una CNN**, solo cambia el modelo y los datos. Esa es la gracia del ciclo de entrenamiento estándar.
- Entrenamos 5 épocas con `Adam` (`lr=1e-3`) y, en cada una, medimos en el conjunto de **prueba** (en este tutorial lo usamos como validación por simplicidad; en un proyecto real se separa un conjunto de validación aparte y el de prueba se reserva para el final).
- **Qué esperar:** la pérdida baja y la exactitud sube en cada época, alcanzando típicamente alrededor del **90 %** de exactitud en pocas épocas. En CPU cada época puede tardar de varios segundos a más de un minuto; en GPU, unos pocos segundos.

In [ ]:
cnn.eval()
imagenes, etiquetas = next(iter(test_loader_fm))
with torch.no_grad():
    probs = torch.softmax(cnn(imagenes[:12].to(device)), dim=1).cpu()
predicciones = probs.argmax(dim=1)
confianzas = probs.max(dim=1).values

fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for ax, im, pred, real, conf in zip(axes.ravel(), imagenes[:12],
                                    predicciones, etiquetas[:12],
                                    confianzas):
    ax.imshow(im.squeeze(0).numpy(), cmap="gray")
    color = "green" if pred == real else "red"
    ax.set_title(f"{clases[pred]} ({conf:.0%})", color=color, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

#### 🔍 Qué hace este código

- `cnn.eval()` activa el modo evaluación (apaga el Dropout) y `torch.no_grad()` evita construir el grafo.
- Tomamos un lote del conjunto de prueba y pasamos las **12 primeras imágenes** por la red (`.to(device)` para moverlas). `torch.softmax(..., dim=1)` convierte los logits en **probabilidades** por clase y `.cpu()` las trae de vuelta para graficar.
- `probs.argmax(dim=1)` es la **clase predicha** y `probs.max(dim=1).values` la **confianza** (la probabilidad de esa clase).
- En el gráfico, el título muestra la predicción y su confianza (`:.0%` la formatea como porcentaje). Es **verde si acertó** y **rojo si falló**. Normalmente la mayoría saldrán en verde; los errores suelen ocurrir entre clases parecidas (camiseta vs camisa).

> 🧠 Con apenas 5 épocas una CNN pequeña supera el ~90% de exactitud. Un MLP equivalente rinde peor y necesita más parámetros.

> ✏️ **Ejercicio 15.1** — Añade un tercer bloque convolucional (`Conv2d(32, 64, 3, padding=1)` + ReLU + MaxPool). ¿Cuál es ahora el tamaño correcto de entrada de la primera `nn.Linear`? (Pista: imprime el shape de `features(x)` con un lote falso en vez de calcularlo a mano.)
>
> *Intenta resolverlo tú primero; la solución está en la celda de código que sigue.*

In [ ]:
# Solución
class CNN3Bloques(nn.Module):
    """CNN con tres bloques convolucionales para imágenes 1x28x28.

    Calcula automáticamente el tamaño de entrada de la primera capa
    lineal pasando una imagen falsa por el extractor de características.

    Args:
        n_clases: número de clases de salida.
    """

    def __init__(self, n_clases=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        with torch.no_grad():
            forma = self.features(torch.zeros(1, 1, 28, 28)).shape
        print("Forma tras el extractor de características:", tuple(forma))
        n_entrada = forma[1] * forma[2] * forma[3]

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_entrada, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_clases),
        )

    def forward(self, x):
        """Devuelve los logits para un lote de imágenes ``x``."""
        return self.classifier(self.features(x))


torch.manual_seed(0)
cnn3 = CNN3Bloques().to(device)
print("Parámetros:", sum(p.numel() for p in cnn3.parameters()))

optimizador = torch.optim.Adam(cnn3.parameters(), lr=1e-3)
tl, ta = entrenar_una_epoca(cnn3, train_loader_fm, loss_fn, optimizador,
                            device)
vl, va = evaluar(cnn3, test_loader_fm, loss_fn, device)
print(f"1 época -> train acc {ta:.3f} | test acc {va:.3f}")

#### 🔍 Qué hace este código

- Añadimos un tercer bloque (`Conv2d(32, 64)` + ReLU + MaxPool). Cada `MaxPool2d(2)` divide el tamaño espacial entre 2 (con división entera): `28 → 14 → 7 → 3` (7/2 = 3,5, que se redondea hacia abajo a 3). Por tanto, la salida del extractor es `(64, 3, 3)` y la primera capa lineal debe recibir **`64 · 3 · 3 = 576`** valores (antes eran 1568).
- En lugar de calcularlo a mano, **lo medimos**: dentro de `__init__` pasamos una imagen de ceros `(1, 1, 28, 28)` por `self.features` (con `torch.no_grad()` para no gastar memoria) y leemos su forma. Así `n_entrada` se calcula solo y el código sigue funcionando aunque cambies las capas. Esta es la técnica recomendada para evitar errores de dimensiones.
- Entrenamos solo 1 época para verificar que todo funciona; ya debería lograr una exactitud razonable (en torno a 80–88 %).

---
# 1️⃣6️⃣ Buenas prácticas y errores comunes

## Reproducibilidad
Los resultados cambian entre ejecuciones por la aleatoriedad (inicialización, barajado, dropout). Para fijar semillas:

In [ ]:
import random


def fijar_semilla(semilla=42):
    """Fija las semillas de Python, NumPy y PyTorch (CPU y GPU).

    Args:
        semilla: entero usado como semilla en todas las bibliotecas.
    """
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(semilla)


fijar_semilla(42)
print("Primera vez :", torch.rand(2))
fijar_semilla(42)
print("Segunda vez :", torch.rand(2))

#### 🔍 Qué hace este código

- Python (`random`), NumPy y PyTorch tienen **generadores de números aleatorios independientes**, y la GPU tiene el suyo. `fijar_semilla` los inicializa todos con el mismo número.
- Al llamar `fijar_semilla(42)` antes de cada `torch.rand(2)`, **los dos resultados son idénticos**: eso es reproducibilidad.
- Nota: incluso con semillas, algunas operaciones en GPU pueden dar diferencias mínimas entre ejecuciones; para reproducibilidad estricta existen opciones adicionales (como `torch.use_deterministic_algorithms(True)`).

## Los errores más frecuentes (y cómo resolverlos)

| Error / síntoma | Causa típica | Solución |
|---|---|---|
| `Expected all tensors to be on the same device` | Mezclaste tensores de CPU y GPU | `.to(device)` en el modelo **y** en los datos |
| `mat1 and mat2 shapes cannot be multiplied` | La entrada de `nn.Linear` no coincide con la salida de la capa anterior | Imprime `x.shape` en cada paso; revisa `in_features` |
| `Expected ... Long ... but got Float` (en `CrossEntropyLoss`) | Las etiquetas no son `int64` | `y = y.long()` |
| `CUDA out of memory` | El batch o el modelo no caben en VRAM | Reduce `batch_size`, usa precisión mixta, libera variables |
| `Can't call numpy() on Tensor that requires grad` | Intentas convertir un tensor con gradiente | `t.detach().cpu().numpy()` |
| La pérdida no baja / es `nan` | `lr` demasiado grande, datos sin normalizar, olvidaste `zero_grad()` | Baja el `lr`, normaliza, revisa el ciclo |
| Resultados raros al evaluar | Olvidaste `model.eval()` | Llama `model.eval()` y usa `torch.no_grad()` |
| El modelo "no aprende" | Softmax antes de `CrossEntropyLoss` (la loss ya lo incluye) | Entrega **logits** crudos |
| `torch.cuda.is_available()` da `False` teniendo GPU NVIDIA | PyTorch instalado sin CUDA, o falta el driver | Ver secciones 2.8 y 2.9 |

## Consejos de depuración
1. **Imprime shapes** (`print(x.shape)`) constantemente al principio.
2. **Sobreajusta un lote pequeño primero**: si el modelo no logra memorizar 10 ejemplos, hay un bug en el código.
3. Revisa el orden del ciclo: `zero_grad → backward → step`.
4. Empieza con un modelo pequeño y complica de a poco.
5. Visualiza tus datos antes de entrenar.

## Un vistazo a temas más avanzados (para explorar luego)
- **Precisión mixta** (`torch.autocast`): entrenar más rápido y con menos memoria.
- **`torch.compile`**: compila el modelo para acelerarlo (PyTorch 2.x).
- **Transfer learning**: reutilizar modelos preentrenados (`torchvision.models`).
- **Schedulers de learning rate** (`torch.optim.lr_scheduler`).
- **Entrenamiento distribuido** (varias GPUs).

In [ ]:
# Técnica de depuración: ¿puede el modelo memorizar un solo lote pequeño?
torch.manual_seed(0)
xb, yb = next(iter(train_loader_fm))
xb, yb = xb[:16].to(device), yb[:16].to(device)

prueba = SimpleCNN().to(device)
opt = torch.optim.Adam(prueba.parameters(), lr=1e-3)
criterio = nn.CrossEntropyLoss()

for _ in range(100):
    loss = criterio(prueba(xb), yb)
    opt.zero_grad()
    loss.backward()
    opt.step()

print(f"Pérdida tras sobreajustar 16 ejemplos: {loss.item():.5f}")

#### 🔍 Qué hace este código

- Tomamos **un solo lote de 16 imágenes** y entrenamos un modelo nuevo **únicamente con esas 16**, 100 veces seguidas.
- Si todo el código está bien (forward, pérdida, backward, optimizador), el modelo **debe ser capaz de memorizar** esos 16 ejemplos y la pérdida debe acercarse a **0** (aquí verás un valor muy pequeño, del orden de 0.001 o menos).
- Si la pérdida **no baja**, hay un error en el ciclo, en las formas, en las etiquetas, etc. Es una prueba rápida (segundos) que ahorra horas.
- Este modelo está en modo entrenamiento (con Dropout activo), por lo que la pérdida final puede ser algo mayor que con `eval()`, pero igualmente debe bajar mucho.

---
# 1️⃣7️⃣ Ejercicios integradores (con solución), *cheat sheet* y próximos pasos

Estos ejercicios combinan varias secciones. Intenta resolver cada uno antes de ver la solución que aparece justo debajo del enunciado.

## Ejercicio 17.1 — Regresión no lineal
Genera datos $y = \sin(x) + \text{ruido}$ en $[-\pi, \pi]$ y entrena un MLP para ajustarlos. Grafica datos vs predicción.

In [ ]:
# Solución 17.1 (usa entrenar_regresion definida en el ejercicio 10.1)
import math

torch.manual_seed(0)
X_seno = torch.linspace(-math.pi, math.pi, 200).unsqueeze(1)
Y_seno = torch.sin(X_seno) + torch.randn(200, 1) * 0.1

modelo_seno = nn.Sequential(
    nn.Linear(1, 32), nn.Tanh(),
    nn.Linear(32, 32), nn.Tanh(),
    nn.Linear(32, 1),
)
opt_seno = torch.optim.Adam(modelo_seno.parameters(), lr=1e-2)
perdidas_seno = entrenar_regresion(modelo_seno, opt_seno, X_seno, Y_seno,
                                   n_epocas=1000)
print(f"Pérdida inicial: {perdidas_seno[0]:.3f} | "
      f"final: {perdidas_seno[-1]:.4f}")

with torch.no_grad():
    pred_seno = modelo_seno(X_seno)

plt.figure(figsize=(7, 4))
plt.scatter(X_seno.numpy(), Y_seno.numpy(), s=8, label="datos")
plt.plot(X_seno.numpy(), pred_seno.numpy(), "r", label="predicción")
plt.legend()
plt.title("MLP ajustando y = sin(x) + ruido")
plt.show()

#### 🔍 Qué hace este código

- Generamos 200 puntos entre $-\pi$ y $\pi$ y calculamos $\sin(x)$ con **ruido pequeño** (0.1).
- La red tiene dos capas ocultas de 32 neuronas con activación **Tanh** (suave, muy adecuada para aproximar curvas como el seno).
- Reutilizamos la función `entrenar_regresion` del ejercicio 10.1 (si saltaste ese ejercicio, ejecuta antes esa celda).
- La pérdida final debería acercarse a `0.01` (la varianza del ruido, $0.1^2$): el modelo aprendió la curva sin memorizar el ruido. El gráfico muestra la línea roja siguiendo la forma del seno.

## Ejercicio 17.2 — Clasificación binaria
Genera el dataset "dos lunas" (a mano, sin scikit-learn), con una red de **1 neurona de salida** y `nn.BCEWithLogitsLoss`. Reporta la exactitud en un conjunto de validación.

In [ ]:
# Solución 17.2
def hacer_lunas(n=400, ruido=0.15):
    """Genera el dataset 'dos lunas' para clasificación binaria.

    Args:
        n: número total de puntos (mitad por luna).
        ruido: desviación estándar del ruido gaussiano.

    Returns:
        Tupla ``(X, y)``: ``X`` de forma (n, 2) e ``y`` de forma (n,)
        con valores 0.0 o 1.0 (``float32``, como pide BCEWithLogitsLoss).
    """
    n_mitad = n // 2
    angulo = torch.linspace(0, math.pi, n_mitad)
    luna_sup = torch.stack([torch.cos(angulo), torch.sin(angulo)], dim=1)
    luna_inf = torch.stack([1 - torch.cos(angulo),
                            0.5 - torch.sin(angulo)], dim=1)
    puntos = torch.cat([luna_sup, luna_inf])
    puntos = puntos + torch.randn(2 * n_mitad, 2) * ruido
    etiquetas = torch.cat([torch.zeros(n_mitad), torch.ones(n_mitad)])
    return puntos, etiquetas


torch.manual_seed(0)
X_l, y_l = hacer_lunas()

# División 75 % / 25 % con una permutación aleatoria
perm = torch.randperm(len(X_l))
corte = int(0.75 * len(X_l))
X_tr_l, y_tr_l = X_l[perm[:corte]], y_l[perm[:corte]]
X_va_l, y_va_l = X_l[perm[corte:]], y_l[perm[corte:]]

modelo_l = nn.Sequential(
    nn.Linear(2, 16), nn.ReLU(),
    nn.Linear(16, 16), nn.ReLU(),
    nn.Linear(16, 1),
)
criterio_bin = nn.BCEWithLogitsLoss()
opt_l = torch.optim.Adam(modelo_l.parameters(), lr=1e-2)

for _ in range(300):
    modelo_l.train()
    logits = modelo_l(X_tr_l).squeeze(1)
    loss = criterio_bin(logits, y_tr_l)
    opt_l.zero_grad()
    loss.backward()
    opt_l.step()


@torch.no_grad()
def exactitud_binaria(modelo, X, y):
    """Calcula la exactitud de un clasificador binario con logits."""
    modelo.eval()
    prediccion = (modelo(X).squeeze(1) > 0).float()
    return (prediccion == y).float().mean().item()


print(f"Exactitud train: {exactitud_binaria(modelo_l, X_tr_l, y_tr_l):.3f}")
print(f"Exactitud val  : {exactitud_binaria(modelo_l, X_va_l, y_va_l):.3f}")

plt.figure(figsize=(5, 4))
plt.scatter(X_l[:, 0].numpy(), X_l[:, 1].numpy(), c=y_l.numpy(), s=10,
            cmap="coolwarm")
plt.title("Dataset de dos lunas")
plt.show()

#### 🔍 Qué hace este código

- **`hacer_lunas`** construye dos semicírculos entrelazados (fórmula equivalente a `make_moons` de scikit-learn) y les suma ruido. Las etiquetas son `0.0` y `1.0` como **`float32`**, porque `BCEWithLogitsLoss` exige etiquetas decimales (a diferencia de `CrossEntropyLoss`, que pide enteros).
- **División:** `torch.randperm(n)` genera una permutación aleatoria de los índices `0..n-1`; los primeros 75 % van a entrenamiento y el resto a validación.
- **Modelo:** termina en `nn.Linear(16, 1)`, **una sola neurona de salida** que entrega un *logit*: positivo significa "clase 1" y negativo "clase 0". La pérdida `BCEWithLogitsLoss` aplica la sigmoide internamente.
- **`squeeze(1)`** convierte la salida `(N, 1)` en `(N,)` para que coincida con la forma de las etiquetas.
- **Entrenamiento** *full-batch* (todos los datos en cada paso): 300 pasos del ciclo estándar.
- **`exactitud_binaria`:** un logit `> 0` equivale a una probabilidad `> 0.5` (sigmoide de 0 es 0.5), así que la clase predicha es `(logit > 0)`. Comparamos con las etiquetas y promediamos.
- **Qué esperar:** exactitud muy alta (típicamente ≥ 0.95) en entrenamiento y validación.

## Ejercicio 17.3 — Comparar optimizadores
Entrena el mismo modelo con SGD, SGD con momentum (0.9) y Adam sobre el problema de las espirales. Grafica las tres curvas de pérdida de entrenamiento.

In [ ]:
# Solución 17.3 (usa train_loader, loss_fn y entrenar_una_epoca)
def crear_optimizador(nombre, parametros):
    """Crea el optimizador pedido con una tasa de aprendizaje fija."""
    if nombre == "SGD":
        return torch.optim.SGD(parametros, lr=0.01)
    if nombre == "SGD + momentum":
        return torch.optim.SGD(parametros, lr=0.01, momentum=0.9)
    return torch.optim.Adam(parametros, lr=0.01)


curvas = {}
for nombre in ["SGD", "SGD + momentum", "Adam"]:
    torch.manual_seed(0)
    modelo_op = crear_mlp(64, True)
    opt_op = crear_optimizador(nombre, modelo_op.parameters())
    perdidas = []
    for _ in range(50):
        tl, _ = entrenar_una_epoca(modelo_op, train_loader, loss_fn,
                                   opt_op, device)
        perdidas.append(tl)
    curvas[nombre] = perdidas
    print(f"{nombre:15s} -> pérdida final de entrenamiento: "
          f"{perdidas[-1]:.3f}")

plt.figure(figsize=(7, 4))
for nombre, perdidas in curvas.items():
    plt.plot(perdidas, label=nombre)
plt.xlabel("Época")
plt.ylabel("Pérdida de entrenamiento")
plt.legend()
plt.title("Comparación de optimizadores")
plt.show()

#### 🔍 Qué hace este código

- `crear_optimizador` devuelve el optimizador según el nombre, todos con `lr = 0.01` para comparar en igualdad de condiciones. **Momentum** hace que el optimizador "acumule inercia" en la dirección en que viene bajando, acelerando el avance y suavizando oscilaciones.
- Para cada uno se crea un modelo **nuevo con la misma semilla** (así todos parten con los mismos pesos) y se entrena 50 épocas, guardando la pérdida de entrenamiento de cada época (`tl, _ = ...` ignora la exactitud).
- Reutilizamos `crear_mlp` (ejercicio 12.1), `train_loader` (sección 12) y `entrenar_una_epoca`.
- **Qué esperar:** `SGD` simple avanza **más lento**; `SGD + momentum` acelera claramente; `Adam` suele ser el que baja más rápido con una tasa de aprendizaje sin ajustar. Por eso Adam es el punto de partida habitual.

## Ejercicio 17.4 — Efecto del `batch_size`
Entrena la CNN con `batch_size` de 32, 128 y 512 (una época cada uno, con un subconjunto de 10.000 imágenes para que sea rápido). ¿Cómo cambian el tiempo por época y la exactitud?

In [ ]:
# Solución 17.4
from torch.utils.data import Subset

subconjunto = Subset(train_data, range(10000))

print(f"{'batch_size':>10s} | {'iteraciones':>11s} | {'tiempo (s)':>10s} "
      f"| {'test acc':>8s}")
for tamano_lote in [32, 128, 512]:
    torch.manual_seed(0)
    loader_bs = DataLoader(subconjunto, batch_size=tamano_lote,
                           shuffle=True)
    cnn_bs = SimpleCNN().to(device)
    opt_bs = torch.optim.Adam(cnn_bs.parameters(), lr=1e-3)

    sincronizar(device)
    inicio = time.perf_counter()
    entrenar_una_epoca(cnn_bs, loader_bs, loss_fn, opt_bs, device)
    sincronizar(device)
    duracion = time.perf_counter() - inicio

    acc = evaluar(cnn_bs, test_loader_fm, loss_fn, device)[1]
    print(f"{tamano_lote:>10d} | {len(loader_bs):>11d} | "
          f"{duracion:>10.2f} | {acc:>8.3f}")

#### 🔍 Qué hace este código

- `Subset(train_data, range(10000))` toma solo las **primeras 10.000 imágenes** del dataset (para que el experimento sea rápido).
- Por cada tamaño de lote: creamos el `DataLoader`, un modelo nuevo (con la misma semilla) y medimos el tiempo de **una época**, usando `sincronizar` (definida en la sección 6) para que la medición sea correcta en GPU. Después evaluamos en el conjunto de prueba completo.
- La tabla muestra el número de iteraciones (`len(loader_bs)`), el tiempo y la exactitud.
- **Qué esperar:** con lotes **pequeños** hay **más iteraciones** (más actualizaciones de pesos por época), por lo que el modelo tiende a aprender más en una época (mayor exactitud), pero cada época suele tardar más en GPU porque se aprovecha menos el paralelismo. Con lotes **grandes** hay menos actualizaciones: la época es más rápida pero, con el mismo número de épocas, la exactitud suele ser menor. Es un compromiso clásico entre velocidad y calidad; en la práctica se ajusta junto con la tasa de aprendizaje.

## Ejercicio 17.5 — Un `Dataset` propio que lee imágenes de carpetas
Escribe un `Dataset` que lea imágenes organizadas en subcarpetas por clase (como `torchvision.datasets.ImageFolder`). Para poder probarlo sin descargar nada, la solución primero crea unas imágenes de ejemplo en una carpeta temporal.

In [ ]:
# Solución 17.5
import tempfile
from pathlib import Path

from PIL import Image

# 1) Crear una carpeta temporal con 2 clases y 5 imágenes aleatorias c/u
raiz = Path(tempfile.mkdtemp())
for nombre_clase in ["gatos", "perros"]:
    (raiz / nombre_clase).mkdir()
    for i in range(5):
        pixeles = np.random.randint(0, 256, (16, 16, 3), dtype=np.uint8)
        Image.fromarray(pixeles).save(raiz / nombre_clase / f"img{i}.png")


class CarpetaImagenes(Dataset):
    """Dataset que lee imágenes PNG organizadas en una carpeta por clase.

    Estructura esperada: ``raiz/<clase>/<archivo>.png``.

    Args:
        raiz: carpeta que contiene una subcarpeta por clase.
        transformacion: función opcional que se aplica a cada imagen.
    """

    def __init__(self, raiz, transformacion=None):
        self.raiz = Path(raiz)
        self.transformacion = transformacion
        carpetas = sorted(p for p in self.raiz.iterdir() if p.is_dir())
        self.clases = [p.name for p in carpetas]
        self.muestras = [
            (ruta, indice)
            for indice, carpeta in enumerate(carpetas)
            for ruta in sorted(carpeta.glob("*.png"))
        ]

    def __len__(self):
        """Devuelve el número total de imágenes."""
        return len(self.muestras)

    def __getitem__(self, idx):
        """Devuelve el par (imagen, índice de clase) número ``idx``."""
        ruta, etiqueta = self.muestras[idx]
        imagen = Image.open(ruta).convert("RGB")
        if self.transformacion is not None:
            imagen = self.transformacion(imagen)
        return imagen, etiqueta


ds_carpetas = CarpetaImagenes(raiz, transformacion=transforms.ToTensor())
print("Clases encontradas:", ds_carpetas.clases)
print("Número de imágenes:", len(ds_carpetas))

imagen, etiqueta = ds_carpetas[0]
print("Primera imagen:", tuple(imagen.shape), "| etiqueta:", etiqueta)

loader_carpetas = DataLoader(ds_carpetas, batch_size=4, shuffle=True)
xb, yb = next(iter(loader_carpetas))
print("Un lote -> imágenes:", tuple(xb.shape), "| etiquetas:", yb.tolist())

#### 🔍 Qué hace este código

1. **Preparación:** `tempfile.mkdtemp()` crea una carpeta temporal. Dentro creamos dos subcarpetas (`gatos`, `perros`) con 5 imágenes PNG de 16×16 píxeles de ruido aleatorio cada una (`np.random.randint(...)` produce los píxeles y `Image.fromarray(...).save(...)` los guarda). Son imágenes falsas, solo para probar el código.
2. **`CarpetaImagenes`** cumple el "contrato" de un `Dataset` (los tres métodos):
   - `__init__`: lista las subcarpetas en orden alfabético (cada una es una clase) y arma `self.muestras`, una lista de pares `(ruta del archivo, índice de la clase)`. Está escrito con una *list comprehension* de dos `for` anidados: recorre cada carpeta y, dentro, cada archivo `.png`.
   - `__len__`: número de imágenes (10).
   - `__getitem__(idx)`: abre el archivo con PIL, lo pasa a RGB, aplica la transformación (aquí `ToTensor`, que lo convierte en tensor `(3, 16, 16)` con valores 0–1) y devuelve `(imagen, etiqueta)`.
3. **Comprobaciones:** debe encontrar las clases `['gatos', 'perros']`, 10 imágenes, y la primera tendrá forma `(3, 16, 16)`. Al envolverlo en un `DataLoader` con `batch_size=4`, un lote tiene forma `(4, 3, 16, 16)` con 4 etiquetas (0 o 1).
- Con imágenes reales, **el código no cambia**: solo apuntarías `raiz` a tu carpeta y quizá añadirías `Resize` a la transformación para unificar tamaños.

## 📋 Cheat sheet
```python
# --- Tensores ---
torch.tensor(data)  torch.zeros(s)  torch.ones(s)  torch.randn(s)  torch.arange(a, b)
t.shape  t.dtype  t.device        t.to(device)   t.to(torch.float32)
t.reshape(..)  t.unsqueeze(d)  t.squeeze()  t.permute(..)  t.T
torch.cat([a, b], dim)  torch.stack([a, b], dim)  a @ b
t.sum(dim)  t.mean(dim)  t.argmax(dim)  t.item()  t.tolist()

# --- NumPy ---
torch.from_numpy(arr)       t.detach().cpu().numpy()

# --- Autograd ---
x = torch.tensor(.., requires_grad=True);  loss.backward();  x.grad;  x.grad.zero_()
with torch.no_grad(): ...      t.detach()

# --- Modelo ---
class Modelo(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa = nn.Linear(a, b)

    def forward(self, x):
        return self.capa(x)

# --- Entrenamiento ---
model.train()
for xb, yb in loader:
    xb, yb = xb.to(device), yb.to(device)
    loss = loss_fn(model(xb), yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# --- Evaluación ---
model.eval()
with torch.no_grad(): ...

# --- Guardar / cargar ---
torch.save(model.state_dict(), "m.pt")
model.load_state_dict(torch.load("m.pt", weights_only=True))

# --- GPU ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device);  x = x.to(device)
```

## 🧭 Próximos pasos sugeridos
1. **Documentación oficial y tutoriales:** https://pytorch.org/tutorials/ (especialmente "Learn the Basics").
2. **Visión por computador avanzada:** *transfer learning* con `torchvision.models` (ResNet, etc.).
3. **Secuencias y lenguaje:** RNN/LSTM y luego **Transformers** (Hugging Face).
4. **Herramientas de organización:** PyTorch Lightning, TensorBoard o Weights & Biases para registrar experimentos.
5. **Optimización:** precisión mixta, `torch.compile`, entrenamiento multi-GPU.

---
### 🎉 ¡Felicitaciones!
Si llegaste hasta aquí, ya entiendes los **tres pilares** de PyTorch —*tensores, autograd y `nn.Module`*— y sabes entrenar y evaluar un modelo completo. Todo lo demás (Transformers, difusión, modelos de lenguaje) se construye sobre exactamente estas mismas ideas.